# COMP8851 — BWGNN Benchmark

## Author Reproduction and Unified Benchmark

**Model:** BWGNN  
**Paper:** Rethinking Graph Neural Networks for Anomaly Detection  
**Primary first dataset:** T-Finance

### Workflow

1. Verify the official BWGNN repository and upstream commit.
2. Inspect the author-requested software environment.
3. Build an isolated BWGNN-compatible environment.
4. Verify T-Finance and T-Social DGL graph structure.
5. Run a 1–2 epoch smoke test.
6. Perform author-mode reproduction.
7. Classify the reproduction evidence.
8. Apply Team Protocol v1 only after author-mode feasibility passes.
9. Run controlled TR40 / TR30 / TR20 / TR10 experiments.
10. Save configuration, environment, metrics and per-epoch timing.

Author-mode and unified results are never mixed.

## Step 6A — Kaggle Runtime Check

The BWGNN notebook uses the same controlled Kaggle hardware as the shared
dataset notebook.

Kaggle may provide two Tesla T4 GPUs, but benchmark code exposes GPU 0 only.

In [1]:
# ============================================================
# STEP 6A — BWGNN RUNTIME CHECK
# ============================================================

import os
import sys
import time
import subprocess

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:6.2f}s] {msg}",
        flush=True
    )

log("START")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Python:", sys.version.replace("\n", " "), flush=True)
print(
    "CUDA_VISIBLE_DEVICES:",
    os.environ["CUDA_VISIBLE_DEVICES"],
    flush=True
)

log("Checking GPU")

result = subprocess.run(
    ["nvidia-smi", "-L"],
    capture_output=True,
    text=True,
    timeout=20
)

print(result.stdout, flush=True)

log("Importing current Kaggle PyTorch")

import torch

print("Current PyTorch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
print("Visible GPUs:", torch.cuda.device_count(), flush=True)

if torch.cuda.is_available():
    print(
        "Visible GPU 0:",
        torch.cuda.get_device_name(0),
        flush=True
    )

log("DONE")

[  0.00s] START
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CUDA_VISIBLE_DEVICES: 0
[  0.00s] Checking GPU
GPU 0: Tesla T4 (UUID: GPU-bb6e2525-baef-c3a1-1723-3fb895da2976)
GPU 1: Tesla T4 (UUID: GPU-8cfe1d11-0900-912b-c822-ef8c4acfe4f7)

[  0.05s] Importing current Kaggle PyTorch
Current PyTorch: 2.10.0+cu128
CUDA available: True
Visible GPUs: 1
Visible GPU 0: Tesla T4
[  4.31s] DONE


## Step 6B — Official BWGNN Repository Identity

This step freezes the identity of the official BWGNN author repository before
any source code is modified.

It records:

- official repository URL
- exact upstream commit hash
- active upstream branch
- latest commit information
- clean Git status
- top-level repository contents
- presence of the expected BWGNN source files

The author repository is cloned into `/kaggle/working`, outside the COMP8851
team repository.

No datasets are required or accessed in this step.

No packages are installed and no author source files are modified.

In [1]:
# ============================================================
# STEP 6B — FREEZE OFFICIAL BWGNN REPOSITORY IDENTITY
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import json
import time

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OFFICIAL_URL = (
    "https://github.com/squareRoot3/"
    "Rethinking-Anomaly-Detection.git"
)

REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

BWGNN_WORK = Path(
    "/kaggle/working/comp8851_bwgnn"
)

EVIDENCE_DIR = (
    BWGNN_WORK /
    "evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — official BWGNN repository identity audit")

print(
    "\nNo dataset input is required for Step 6B.",
    flush=True
)


# ------------------------------------------------------------
# 1. Remove only a previous Kaggle temporary clone
# ------------------------------------------------------------

if REPO_DIR.exists():

    log(
        "Previous /kaggle/working BWGNN clone found."
    )

    log(
        "Removing temporary clone so this audit starts clean..."
    )

    shutil.rmtree(REPO_DIR)

    log("Previous temporary clone removed.")


# ------------------------------------------------------------
# 2. Clone official repository
# ------------------------------------------------------------

log("Cloning official BWGNN repository...")

clone = subprocess.run(
    [
        "git",
        "clone",
        OFFICIAL_URL,
        str(REPO_DIR)
    ],
    capture_output=True,
    text=True,
    timeout=120
)

if clone.stdout:
    print(
        "\n===== GIT CLONE STDOUT =====",
        flush=True
    )
    print(clone.stdout, flush=True)

if clone.stderr:
    print(
        "\n===== GIT CLONE STDERR =====",
        flush=True
    )
    print(clone.stderr, flush=True)

if clone.returncode != 0:

    raise RuntimeError(
        "Official BWGNN repository clone failed. "
        f"Return code: {clone.returncode}"
    )

log("Clone completed successfully.")


# ------------------------------------------------------------
# 3. Git helper
# ------------------------------------------------------------

def git(*args):

    result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            *args
        ],
        capture_output=True,
        text=True,
        timeout=30
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(args)}\n"
            f"{result.stderr}"
        )

    return result.stdout.strip()


# ------------------------------------------------------------
# 4. Read exact upstream identity
# ------------------------------------------------------------

log("Reading exact upstream Git identity...")

remote_url = git(
    "remote",
    "get-url",
    "origin"
)

commit_hash = git(
    "rev-parse",
    "HEAD"
)

branch_name = git(
    "branch",
    "--show-current"
)

status_porcelain = git(
    "status",
    "--porcelain"
)

latest_commit = git(
    "log",
    "-1",
    "--pretty=format:%H%n%an%n%ae%n%ad%n%s",
    "--date=iso-strict"
)


print(
    "\n===== BWGNN UPSTREAM IDENTITY =====",
    flush=True
)

print(
    "Repository URL :",
    remote_url,
    flush=True
)

print(
    "Commit SHA     :",
    commit_hash,
    flush=True
)

print(
    "Branch         :",
    branch_name,
    flush=True
)

print(
    "Git status     :",
    "CLEAN"
    if not status_porcelain
    else "NOT CLEAN",
    flush=True
)


print(
    "\n===== LATEST UPSTREAM COMMIT =====",
    flush=True
)

print(
    latest_commit,
    flush=True
)


# ------------------------------------------------------------
# 5. Inspect repository files
# ------------------------------------------------------------

log("Inspecting top-level repository files...")

top_level = sorted(
    REPO_DIR.iterdir(),
    key=lambda p: p.name.lower()
)

print(
    "\n===== TOP-LEVEL REPOSITORY CONTENTS =====",
    flush=True
)

for path in top_level:

    kind = (
        "DIR "
        if path.is_dir()
        else "FILE"
    )

    print(
        f"{kind} | {path.name}",
        flush=True
    )


# ------------------------------------------------------------
# 6. Verify expected author source files
# ------------------------------------------------------------

expected_files = [
    "BWGNN.py",
    "dataset.py",
    "main.py",
    "readme.md"
]

print(
    "\n===== EXPECTED BWGNN SOURCE FILE CHECK =====",
    flush=True
)

expected_results = {}

for filename in expected_files:

    path = REPO_DIR / filename

    exists = path.is_file()

    expected_results[filename] = exists

    print(
        f"{'PASS' if exists else 'FAIL'} | {filename}",
        flush=True
    )


if not all(expected_results.values()):

    raise RuntimeError(
        "One or more expected BWGNN author files "
        "are missing. Stop before environment setup."
    )


# ------------------------------------------------------------
# 7. Save frozen upstream identity
# ------------------------------------------------------------

log("Saving upstream identity evidence...")

identity_record = {

    "model": "BWGNN",

    "official_repository": remote_url,

    "upstream_commit": commit_hash,

    "upstream_branch": branch_name,

    "git_status": (
        "CLEAN"
        if not status_porcelain
        else "NOT_CLEAN"
    ),

    "expected_source_files": expected_results,

    "cloned_path": str(REPO_DIR),

    "recorded_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    )
}


identity_path = (
    EVIDENCE_DIR /
    "bwgnn_upstream_identity.json"
)

with identity_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        identity_record,
        f,
        indent=2
    )


files_path = (
    EVIDENCE_DIR /
    "bwgnn_upstream_files.txt"
)

with files_path.open(
    "w",
    encoding="utf-8"
) as f:

    for path in top_level:

        kind = (
            "DIR"
            if path.is_dir()
            else "FILE"
        )

        f.write(
            f"{kind}\t{path.name}\n"
        )


print(
    "\n===== SAVED REPOSITORY EVIDENCE =====",
    flush=True
)

print(
    identity_path,
    flush=True
)

print(
    files_path,
    flush=True
)


# ------------------------------------------------------------
# 8. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6B GATE =====",
    flush=True
)

if (
    not status_porcelain
    and all(expected_results.values())
):

    print(
        "PASS — official BWGNN repository cloned, "
        "identity frozen, and source tree is clean.",
        flush=True
    )

else:

    print(
        "FAIL — repository identity gate not satisfied.",
        flush=True
    )


log("DONE — Step 6B complete")

[   0.00s] START — official BWGNN repository identity audit

No dataset input is required for Step 6B.
[   0.00s] Cloning official BWGNN repository...

===== GIT CLONE STDERR =====
Cloning into '/kaggle/working/Rethinking-Anomaly-Detection'...

[   0.89s] Clone completed successfully.
[   0.89s] Reading exact upstream Git identity...

===== BWGNN UPSTREAM IDENTITY =====
Repository URL : https://github.com/squareRoot3/Rethinking-Anomaly-Detection.git
Commit SHA     : de0631f039bbd19c1890b483cc01f1007f596af7
Branch         : master
Git status     : CLEAN

===== LATEST UPSTREAM COMMIT =====
de0631f039bbd19c1890b483cc01f1007f596af7
DSAIL
dsailathkust@163.com
2024-06-25T17:30:37+08:00
Update readme.md
[   0.91s] Inspecting top-level repository files...

===== TOP-LEVEL REPOSITORY CONTENTS =====
DIR  | .git
FILE | BWGNN.py
FILE | dataset.py
FILE | main.py
FILE | readme.md

===== EXPECTED BWGNN SOURCE FILE CHECK =====
PASS | BWGNN.py
PASS | dataset.py
PASS | main.py
PASS | readme.md
[   0.92s

## Step 6C — Author Environment and Code-Path Audit

This step inspects the exact frozen BWGNN repository commit recorded in
Step 6B.

No packages are installed and no source code is modified.

The audit records:

- dependency versions stated by the author repository
- whether a Python version is explicitly specified
- Python package imports used by the source code
- supported dataset names
- dataset file/path expectations
- command-line arguments and training entry point
- author-provided example commands

The output from this step will be used to design the isolated BWGNN
environment.

No benchmark datasets are required yet.

In [2]:
# ============================================================
# STEP 6C — AUTHOR ENVIRONMENT AND CODE-PATH AUDIT
# ============================================================

from pathlib import Path
import hashlib
import json
import re
import time

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — BWGNN author environment/code audit")

print(
    "\nNo dataset input is required for Step 6C.",
    flush=True
)


# ------------------------------------------------------------
# 1. Required files
# ------------------------------------------------------------

FILES = {
    "readme": REPO_DIR / "readme.md",
    "main": REPO_DIR / "main.py",
    "dataset": REPO_DIR / "dataset.py",
    "model": REPO_DIR / "BWGNN.py",
}

log("Checking required repository files...")

for name, path in FILES.items():

    if not path.exists():
        raise FileNotFoundError(path)

    print(
        f"PASS | {name:<8} | "
        f"{path.name:<12} | "
        f"{path.stat().st_size:,} bytes",
        flush=True
    )


# ------------------------------------------------------------
# 2. Read source files
# ------------------------------------------------------------

log("Reading repository text files...")

texts = {}

for name, path in FILES.items():

    texts[name] = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

log("Repository files read successfully.")


# ------------------------------------------------------------
# 3. SHA-256 of source files
# ------------------------------------------------------------

print(
    "\n===== SOURCE FILE SHA-256 =====",
    flush=True
)

source_hashes = {}

for name, path in FILES.items():

    digest = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    source_hashes[path.name] = digest

    print(
        f"{path.name:<12} : {digest}",
        flush=True
    )


# ------------------------------------------------------------
# 4. README — environment/dependency evidence
# ------------------------------------------------------------

log("Extracting environment-related README lines...")

README_KEYWORDS = [
    "pytorch",
    "torch",
    "dgl",
    "python",
    "sympy",
    "argparse",
    "sklearn",
    "scikit",
    "cuda",
    "requirement",
    "dependency",
]

print(
    "\n===== README — ENVIRONMENT / DEPENDENCY LINES =====",
    flush=True
)

readme_env_lines = []

for lineno, line in enumerate(
    texts["readme"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in README_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        readme_env_lines.append(record)

        print(record, flush=True)


if not readme_env_lines:
    print(
        "No explicit environment/dependency lines found.",
        flush=True
    )


# ------------------------------------------------------------
# 5. Explicit Python-version check
# ------------------------------------------------------------

log("Checking whether author explicitly specifies Python version...")

python_version_pattern = re.compile(
    r"python\s*(?:==|=|>=|<=|>|<)?\s*"
    r"([23](?:\.\d+){1,2})",
    re.IGNORECASE
)

python_version_matches = (
    python_version_pattern.findall(
        texts["readme"]
    )
)

print(
    "\n===== AUTHOR PYTHON VERSION CHECK =====",
    flush=True
)

if python_version_matches:

    print(
        "Explicit Python version reference(s):",
        sorted(set(python_version_matches)),
        flush=True
    )

else:

    print(
        "NOT SPECIFIED — no explicit Python version "
        "was detected in readme.md.",
        flush=True
    )


# ------------------------------------------------------------
# 6. Imports actually used by source
# ------------------------------------------------------------

log("Extracting imports from Python source files...")

import_pattern = re.compile(
    r"^\s*(?:from\s+([A-Za-z0-9_\.]+)\s+import|"
    r"import\s+([A-Za-z0-9_\.]+))",
    re.MULTILINE
)

print(
    "\n===== SOURCE IMPORTS =====",
    flush=True
)

imports_by_file = {}

for key in ["main", "dataset", "model"]:

    imports = set()

    for match in import_pattern.finditer(
        texts[key]
    ):

        module = (
            match.group(1)
            or match.group(2)
        )

        if module:
            imports.add(module.split(".")[0])

    imports_by_file[
        FILES[key].name
    ] = sorted(imports)

    print(
        f"\n[{FILES[key].name}]",
        flush=True
    )

    for module in sorted(imports):
        print(
            f"  {module}",
            flush=True
        )


# ------------------------------------------------------------
# 7. Dataset-related README evidence
# ------------------------------------------------------------

log("Extracting dataset-related README lines...")

DATASET_KEYWORDS = [
    "yelp",
    "amazon",
    "tfinance",
    "t-finance",
    "tsocial",
    "t-social",
    "dataset",
    "google drive",
]

print(
    "\n===== README — DATASET / RUN INSTRUCTIONS =====",
    flush=True
)

readme_dataset_lines = []

for lineno, line in enumerate(
    texts["readme"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in DATASET_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        readme_dataset_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 8. Dataset loader code paths
# ------------------------------------------------------------

log("Inspecting dataset.py dataset/path logic...")

CODE_DATASET_KEYWORDS = [
    "yelp",
    "amazon",
    "tfinance",
    "tsocial",
    "load_graph",
    "fraud",
    "dataset/",
    "./dataset",
]

print(
    "\n===== dataset.py — DATASET/PATH LINES =====",
    flush=True
)

dataset_code_lines = []

for lineno, line in enumerate(
    texts["dataset"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in CODE_DATASET_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        dataset_code_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 9. main.py CLI arguments
# ------------------------------------------------------------

log("Inspecting main.py command-line arguments...")

print(
    "\n===== main.py — CLI ARGUMENTS =====",
    flush=True
)

cli_lines = []

for lineno, line in enumerate(
    texts["main"].splitlines(),
    start=1
):

    if (
        "add_argument" in line
        or "ArgumentParser" in line
    ):

        record = f"L{lineno:03d}: {line}"

        cli_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 10. Training / optimizer / split evidence
# ------------------------------------------------------------

log("Inspecting training-control code...")

TRAIN_KEYWORDS = [
    "adam",
    "optimizer",
    "train_test_split",
    "random_state",
    "train_ratio",
    "threshold",
    "epoch",
    "auc",
    "f1",
    "recall",
    "precision",
]

print(
    "\n===== main.py — TRAINING / SPLIT / METRIC LINES =====",
    flush=True
)

training_lines = []

for lineno, line in enumerate(
    texts["main"].splitlines(),
    start=1
):

    lower = line.lower()

    if any(
        keyword in lower
        for keyword in TRAIN_KEYWORDS
    ):

        record = f"L{lineno:03d}: {line}"

        training_lines.append(record)

        print(record, flush=True)


# ------------------------------------------------------------
# 11. Save audit evidence
# ------------------------------------------------------------

log("Saving Step 6C evidence...")

audit = {

    "model": "BWGNN",

    "upstream_commit":
        "de0631f039bbd19c1890b483cc01f1007f596af7",

    "source_hashes":
        source_hashes,

    "explicit_python_versions_detected":
        sorted(
            set(python_version_matches)
        ),

    "python_version_status":
        (
            "EXPLICITLY_SPECIFIED"
            if python_version_matches
            else "NOT_SPECIFIED"
        ),

    "imports":
        imports_by_file,

    "readme_environment_lines":
        readme_env_lines,

    "readme_dataset_lines":
        readme_dataset_lines,

    "dataset_code_lines":
        dataset_code_lines,

    "cli_lines":
        cli_lines,

    "training_lines":
        training_lines,
}


audit_path = (
    EVIDENCE_DIR /
    "bwgnn_author_environment_audit.json"
)

with audit_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )


print(
    "\n===== SAVED STEP 6C EVIDENCE =====",
    flush=True
)

print(
    audit_path,
    flush=True
)


# ------------------------------------------------------------
# 12. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6C GATE =====",
    flush=True
)

print(
    "PASS — author environment declarations, "
    "dataset paths and entry-point controls "
    "were inspected without modifying the repository.",
    flush=True
)

log("DONE — Step 6C complete")

[   0.00s] START — BWGNN author environment/code audit

No dataset input is required for Step 6C.
[   0.00s] Checking required repository files...
PASS | readme   | readme.md    | 2,105 bytes
PASS | main     | main.py      | 5,621 bytes
PASS | dataset  | dataset.py   | 2,841 bytes
PASS | model    | BWGNN.py     | 7,109 bytes
[   0.01s] Reading repository text files...
[   0.01s] Repository files read successfully.

===== SOURCE FILE SHA-256 =====
readme.md    : 45b5299c07f3efab1c9b8c3fbc30efd64a1b2836efe890b1b34ceff97f84a886
main.py      : 1642e10e5c799ebd303bb20d66a991a55363276b35855996f5615eba0fd1ea13
dataset.py   : 229b5e5101c63879264802f909e64396a49be694a8bb4815dfea201f9b78d1fb
BWGNN.py     : 0ac45b1104f80fa17c50cd5e3bf9aa6241e008e6b57fdab9366ec69f80cb2f84
[   0.02s] Extracting environment-related README lines...

===== README — ENVIRONMENT / DEPENDENCY LINES =====
L014: - pytorch 1.9.0
L015: - dgl 0.8.1
L016: - sympy
L017: - argparse
L018: - sklearn
L031: python main.py --dataset 

## Step 6D — BWGNN Environment Feasibility Audit

The frozen author repository requires:

- PyTorch 1.9.0
- DGL 0.8.1
- sympy
- argparse
- sklearn

The author repository does not specify a Python version.

The current Kaggle base runtime uses Python 3.12, which is too new for the
official PyTorch 1.9.0 and DGL 0.8.1 wheel combination required for a close
author-compatible reproduction.

A candidate isolated environment is therefore:

- Python 3.9
- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1
- NVIDIA T4, GPU 0 only

Python 3.9 is an operational compatibility choice, not an author-stated
requirement.

This step does not install or modify any package. It checks which environment
creation tools are available in the current Kaggle runtime and records GPU,
driver, disk and Python information before environment construction.

In [3]:
# ============================================================
# STEP 6D — ENVIRONMENT FEASIBILITY AUDIT
# ============================================================

import os
import sys
import shutil
import subprocess
import platform
import json
import time
from pathlib import Path
from datetime import datetime, timezone

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


log("START — BWGNN environment feasibility audit")

print(
    "\nNo datasets are required for Step 6D.",
    flush=True
)


# ------------------------------------------------------------
# 1. Current base runtime
# ------------------------------------------------------------

log("Recording current Kaggle base runtime...")

print("\n===== CURRENT KAGGLE BASE =====", flush=True)

print(
    "Python executable :",
    sys.executable,
    flush=True
)

print(
    "Python version    :",
    sys.version.replace("\n", " "),
    flush=True
)

print(
    "Platform          :",
    platform.platform(),
    flush=True
)

print(
    "Architecture      :",
    platform.machine(),
    flush=True
)

print(
    "CUDA_VISIBLE_DEVICES:",
    os.environ.get("CUDA_VISIBLE_DEVICES"),
    flush=True
)


# ------------------------------------------------------------
# 2. Environment manager/tool availability
# ------------------------------------------------------------

log("Checking available environment-management tools...")

tools = [
    "conda",
    "mamba",
    "micromamba",
    "uv",
    "python3.9",
    "python3.10",
    "pip",
    "git"
]

tool_paths = {}

print(
    "\n===== ENVIRONMENT TOOL AVAILABILITY =====",
    flush=True
)

for tool in tools:

    path = shutil.which(tool)

    tool_paths[tool] = path

    print(
        f"{tool:<12} : "
        f"{path if path else 'NOT FOUND'}",
        flush=True
    )


# ------------------------------------------------------------
# 3. GPU + driver
# ------------------------------------------------------------

log("Reading NVIDIA driver/GPU information...")

try:

    gpu_result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu="
            "index,name,driver_version,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(
        "\n===== NVIDIA DRIVER / GPU =====",
        flush=True
    )

    if gpu_result.stdout:
        print(
            gpu_result.stdout.strip(),
            flush=True
        )

    if gpu_result.stderr:
        print(
            gpu_result.stderr.strip(),
            flush=True
        )

except subprocess.TimeoutExpired:

    raise RuntimeError(
        "nvidia-smi timed out after 20 seconds."
    )


# ------------------------------------------------------------
# 4. Current PyTorch details
# ------------------------------------------------------------

log("Reading current PyTorch details...")

import torch

print(
    "\n===== CURRENT PYTORCH =====",
    flush=True
)

print(
    "PyTorch version  :",
    torch.__version__,
    flush=True
)

print(
    "Torch CUDA build :",
    torch.version.cuda,
    flush=True
)

print(
    "CUDA available   :",
    torch.cuda.is_available(),
    flush=True
)

print(
    "Visible GPUs     :",
    torch.cuda.device_count(),
    flush=True
)

if torch.cuda.is_available():

    print(
        "Visible GPU 0    :",
        torch.cuda.get_device_name(0),
        flush=True
    )


# ------------------------------------------------------------
# 5. Disk availability
# ------------------------------------------------------------

log("Checking /kaggle/working disk availability...")

usage = shutil.disk_usage(
    "/kaggle/working"
)

print(
    "\n===== /KAGGLE/WORKING DISK =====",
    flush=True
)

print(
    f"Total : {usage.total / (1024**3):.2f} GiB",
    flush=True
)

print(
    f"Used  : {usage.used / (1024**3):.2f} GiB",
    flush=True
)

print(
    f"Free  : {usage.free / (1024**3):.2f} GiB",
    flush=True
)


# ------------------------------------------------------------
# 6. Candidate environment record
# ------------------------------------------------------------

candidate = {

    "python": "3.9",

    "python_basis":
        "Compatibility choice; author did not specify Python",

    "torch": "1.9.0+cu111",

    "torch_basis":
        "Author requested PyTorch 1.9.0; CUDA 11.1 "
        "selected as compatible official GPU build",

    "dgl": "0.8.1 CUDA 11.1",

    "dgl_basis":
        "Author requested DGL 0.8.1; CUDA 11.1 "
        "selected to match PyTorch build",

    "gpu_policy": "NVIDIA T4, GPU 0 only"
}


print(
    "\n===== CANDIDATE ISOLATED BWGNN ENVIRONMENT =====",
    flush=True
)

for key, value in candidate.items():

    print(
        f"{key:<15}: {value}",
        flush=True
    )


# ------------------------------------------------------------
# 7. Determine available construction path
# ------------------------------------------------------------

print(
    "\n===== ENVIRONMENT CONSTRUCTION OPTIONS =====",
    flush=True
)

if tool_paths["conda"]:

    print(
        "AVAILABLE: conda-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["mamba"]:

    print(
        "AVAILABLE: mamba-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["micromamba"]:

    print(
        "AVAILABLE: micromamba-based isolated Python 3.9 environment",
        flush=True
    )

if tool_paths["uv"]:

    print(
        "AVAILABLE: uv-managed isolated Python environment",
        flush=True
    )

if tool_paths["python3.9"]:

    print(
        "AVAILABLE: system Python 3.9 executable",
        flush=True
    )

if not any(
    [
        tool_paths["conda"],
        tool_paths["mamba"],
        tool_paths["micromamba"],
        tool_paths["uv"],
        tool_paths["python3.9"],
    ]
):

    print(
        "NO READY ISOLATED-PYTHON TOOL FOUND.",
        flush=True
    )

    print(
        "Do not install old PyTorch into the Python 3.12 base.",
        flush=True
    )


# ------------------------------------------------------------
# 8. Save audit
# ------------------------------------------------------------

log("Saving Step 6D feasibility evidence...")

record = {

    "recorded_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "base_python":
        sys.version.replace("\n", " "),

    "base_python_executable":
        sys.executable,

    "base_torch":
        torch.__version__,

    "base_torch_cuda":
        torch.version.cuda,

    "cuda_visible_devices":
        os.environ.get("CUDA_VISIBLE_DEVICES"),

    "tool_paths":
        tool_paths,

    "disk_free_bytes":
        usage.free,

    "candidate_environment":
        candidate
}


output_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_feasibility.json"
)

with output_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        record,
        f,
        indent=2
    )


print(
    "\n===== SAVED STEP 6D EVIDENCE =====",
    flush=True
)

print(
    output_path,
    flush=True
)


# ------------------------------------------------------------
# 9. Gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6D GATE =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no packages were installed or modified.",
    flush=True
)

log("DONE — Step 6D complete")

[   0.00s] START — BWGNN environment feasibility audit

No datasets are required for Step 6D.
[   0.00s] Recording current Kaggle base runtime...

===== CURRENT KAGGLE BASE =====
Python executable : /usr/bin/python3
Python version    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Architecture      : x86_64
CUDA_VISIBLE_DEVICES: None
[   0.01s] Checking available environment-management tools...

===== ENVIRONMENT TOOL AVAILABILITY =====
conda        : NOT FOUND
mamba        : /usr/local/bin/mamba
micromamba   : NOT FOUND
uv           : /usr/local/bin/uv
python3.9    : NOT FOUND
python3.10   : /usr/bin/python3.10
pip          : /usr/local/bin/pip
git          : /usr/bin/git
[   0.02s] Reading NVIDIA driver/GPU information...

===== NVIDIA DRIVER / GPU =====
0, Tesla T4, 580.159.04, 15360 MiB
1, Tesla T4, 580.159.04, 15360 MiB
[   0.06s] Reading current PyTorch details...

===== CURRENT PYTORCH =====
PyTorch version  : 2.10.0+

## Step 6E — Build the Isolated BWGNN Environment

The author repository explicitly requests PyTorch 1.9.0 and DGL 0.8.1,
but does not specify a Python version.

Python 3.9 is therefore used as a compatibility choice, not as an
author-stated requirement.

The isolated environment will target:

- Python 3.9
- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1
- scikit-learn
- SciPy
- SymPy
- NumPy 1.x
- NVIDIA T4
- GPU 0 only

The existing Kaggle Python 3.12 / PyTorch 2.10 environment will not be
modified.

All BWGNN subprocesses explicitly receive `CUDA_VISIBLE_DEVICES=0`.

In [4]:
# ============================================================
# STEP 6E.1 — GPU POLICY + MAMBA CHECK
# ============================================================

import os
import shutil
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


log("START — preparing isolated-environment policy")

# ------------------------------------------------------------
# Explicit policy for every BWGNN subprocess
# ------------------------------------------------------------

BWGNN_SUBPROCESS_ENV = os.environ.copy()

BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
BWGNN_SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"

print("\n===== BWGNN SUBPROCESS POLICY =====", flush=True)

print(
    "CUDA_VISIBLE_DEVICES =",
    BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"],
    flush=True
)

print(
    "DGLBACKEND           =",
    BWGNN_SUBPROCESS_ENV["DGLBACKEND"],
    flush=True
)


# ------------------------------------------------------------
# Isolated environment location
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

BWGNN_ENV.parent.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Environment path     =",
    BWGNN_ENV,
    flush=True
)


# ------------------------------------------------------------
# Mamba check
# ------------------------------------------------------------

MAMBA = shutil.which("mamba")

if not MAMBA:
    raise RuntimeError(
        "mamba disappeared from the Kaggle runtime."
    )

log(f"mamba located at: {MAMBA}")

log("Reading mamba version...")

result = subprocess.run(
    [MAMBA, "--version"],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== MAMBA VERSION =====",
    flush=True
)

print(
    result.stdout.strip()
    or result.stderr.strip(),
    flush=True
)

if result.returncode != 0:
    raise RuntimeError("mamba version check failed.")


print(
    "\n===== STEP 6E.1 GATE =====",
    flush=True
)

print(
    "PASS — GPU0 policy defined and mamba is available.",
    flush=True
)

log("DONE — Step 6E.1")

[   0.00s] START — preparing isolated-environment policy

===== BWGNN SUBPROCESS POLICY =====
CUDA_VISIBLE_DEVICES = 0
DGLBACKEND           = pytorch
Environment path     = /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
[   0.01s] mamba located at: /usr/local/bin/mamba
[   0.01s] Reading mamba version...

===== MAMBA VERSION =====
0.11.3

===== STEP 6E.1 GATE =====
PASS — GPU0 policy defined and mamba is available.
[   0.42s] DONE — Step 6E.1


### Step 6E.2 — Create Python 3.9 Environment

A fresh Python 3.9 environment is created under `/kaggle/working`.

Only Python and pip are installed at this stage.

No PyTorch, DGL or model dependencies are installed yet.

Progress from mamba is streamed to the notebook while the environment is
created.

In [5]:
# ============================================================
# STEP 6E.2 — CREATE ISOLATED PYTHON 3.9 ENVIRONMENT
# ============================================================

import subprocess
import shutil
import time

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


def run_live(command, env=None, timeout=900):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    start = time.perf_counter()

    try:

        for line in process.stdout:
            print(line.rstrip(), flush=True)

            if time.perf_counter() - start > timeout:
                process.kill()
                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:
        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


log("START — creating BWGNN Python 3.9 environment")

# Remove only an incomplete BWGNN environment from a failed attempt.
if BWGNN_ENV.exists():

    log(
        "Existing BWGNN environment path found. "
        "Removing it for a clean creation."
    )

    shutil.rmtree(BWGNN_ENV)

    log("Previous environment removed.")


log("Starting mamba environment creation...")

run_live(
    [
        MAMBA,
        "create",
        "-y",
        "-p",
        BWGNN_ENV,
        "-c",
        "conda-forge",
        "python=3.9",
        "pip"
    ],
    env=BWGNN_SUBPROCESS_ENV,
    timeout=900
)


PYTHON39 = (
    BWGNN_ENV /
    "bin/python"
)

PIP39 = (
    BWGNN_ENV /
    "bin/pip"
)


if not PYTHON39.exists():
    raise RuntimeError(
        "Python executable was not created."
    )


log("Checking isolated Python...")

result = subprocess.run(
    [
        str(PYTHON39),
        "--version"
    ],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== ISOLATED PYTHON =====",
    flush=True
)

print(
    result.stdout.strip()
    or result.stderr.strip(),
    flush=True
)

print(
    "Executable:",
    PYTHON39,
    flush=True
)


print(
    "\n===== STEP 6E.2 GATE =====",
    flush=True
)

if result.returncode == 0 and "Python 3.9" in (
    result.stdout + result.stderr
):

    print(
        "PASS — isolated Python 3.9 environment created.",
        flush=True
    )

else:

    raise RuntimeError(
        "Python 3.9 environment verification failed."
    )


log("DONE — Step 6E.2")

[   0.00s] START — creating BWGNN Python 3.9 environment
[   0.00s] Starting mamba environment creation...

COMMAND: /usr/local/bin/mamba create -y -p /kaggle/working/comp8851_bwgnn/envs/bwgnn-author -c conda-forge python=3.9 pip
usage: mamba [-h] [--version] [--slow SLOW] [--enable-coverage]
             [--coverage-file COVERAGE_FILE] [--format FORMAT] [--no-color]
             [--tags TAGS]
             [specs ...]
mamba: error: unrecognized arguments: -y -p /kaggle/working/comp8851_bwgnn/envs/bwgnn-author -c conda-forge python=3.9 pip


RuntimeError: Command failed with return code 2

### Step 6E.2 — Environment Creation Correction

The initial environment-creation attempt detected an executable named
`mamba`, but runtime inspection showed that it is not the Conda-compatible
Mamba package manager.

The command failed before creating or modifying the BWGNN environment.

The environment-construction method is therefore changed to `uv`, which is
available in the Kaggle runtime and can download a requested Python version
when it is not already installed.

Python 3.9 remains an operational compatibility choice because the BWGNN
author repository does not specify a Python version.

No author dependency version is changed by this correction.

In [6]:
# ============================================================
# STEP 6E.2 — CORRECTED
# CREATE ISOLATED PYTHON 3.9 ENVIRONMENT WITH UV
# ============================================================

import os
import shutil
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# 1. Re-declare paths and execution policy
# ------------------------------------------------------------

UV = shutil.which("uv")

if not UV:
    raise RuntimeError(
        "uv is no longer available in this Kaggle runtime."
    )

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

BWGNN_ENV.parent.mkdir(
    parents=True,
    exist_ok=True
)

BWGNN_SUBPROCESS_ENV = os.environ.copy()

# Controlled benchmark policy
BWGNN_SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
BWGNN_SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"


# ------------------------------------------------------------
# 2. Live subprocess helper
# ------------------------------------------------------------

def run_live(command, env=None, timeout=900):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )

    command_start = time.perf_counter()

    try:

        for line in process.stdout:

            print(
                line.rstrip(),
                flush=True
            )

            elapsed = (
                time.perf_counter()
                - command_start
            )

            if elapsed > timeout:

                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:

        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


# ------------------------------------------------------------
# 3. Start audit
# ------------------------------------------------------------

log("START — corrected Python 3.9 environment creation")

print(
    "\n===== ENVIRONMENT CREATION METHOD =====",
    flush=True
)

print(
    "Tool        : uv",
    flush=True
)

print(
    "uv path     :",
    UV,
    flush=True
)

print(
    "Target path :",
    BWGNN_ENV,
    flush=True
)

print(
    "GPU policy  : CUDA_VISIBLE_DEVICES=0",
    flush=True
)


# ------------------------------------------------------------
# 4. Record uv version
# ------------------------------------------------------------

log("Checking uv version...")

uv_version = subprocess.run(
    [UV, "--version"],
    capture_output=True,
    text=True,
    timeout=20
)

print(
    "\n===== UV VERSION =====",
    flush=True
)

print(
    uv_version.stdout.strip()
    or uv_version.stderr.strip(),
    flush=True
)

if uv_version.returncode != 0:
    raise RuntimeError("uv version check failed.")


# ------------------------------------------------------------
# 5. Remove incomplete environment if one exists
# ------------------------------------------------------------

if BWGNN_ENV.exists():

    log(
        "Existing environment directory detected."
    )

    log(
        "Removing incomplete environment before clean creation..."
    )

    shutil.rmtree(BWGNN_ENV)

    log("Previous environment directory removed.")


# ------------------------------------------------------------
# 6. Create Python 3.9 venv
#
# uv will download Python 3.9 automatically if needed.
# --seed installs pip/setuptools/wheel into the venv.
# ------------------------------------------------------------

log(
    "Creating isolated Python 3.9 environment with uv..."
)

run_live(
    [
        UV,
        "venv",
        "--python",
        "3.9",
        "--seed",
        str(BWGNN_ENV)
    ],
    env=BWGNN_SUBPROCESS_ENV,
    timeout=900
)


# ------------------------------------------------------------
# 7. Verify executables
# ------------------------------------------------------------

PYTHON39 = (
    BWGNN_ENV /
    "bin/python"
)

PIP39 = (
    BWGNN_ENV /
    "bin/pip"
)

print(
    "\n===== CREATED EXECUTABLES =====",
    flush=True
)

print(
    "Python:",
    PYTHON39,
    "| exists:",
    PYTHON39.exists(),
    flush=True
)

print(
    "pip   :",
    PIP39,
    "| exists:",
    PIP39.exists(),
    flush=True
)

if not PYTHON39.exists():

    raise RuntimeError(
        "Python executable was not created."
    )


# ------------------------------------------------------------
# 8. Exact Python version
# ------------------------------------------------------------

log("Reading exact isolated Python version...")

python_check = subprocess.run(
    [
        str(PYTHON39),
        "--version"
    ],
    capture_output=True,
    text=True,
    timeout=20,
    env=BWGNN_SUBPROCESS_ENV
)

python_version_text = (
    python_check.stdout.strip()
    or python_check.stderr.strip()
)

print(
    "\n===== ISOLATED PYTHON =====",
    flush=True
)

print(
    "Version   :",
    python_version_text,
    flush=True
)

print(
    "Executable:",
    PYTHON39,
    flush=True
)


# ------------------------------------------------------------
# 9. Verify interpreter internals
# ------------------------------------------------------------

log("Checking isolated interpreter details...")

detail_check = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        (
            "import sys, platform; "
            "print('sys.version =', sys.version); "
            "print('executable =', sys.executable); "
            "print('platform =', platform.platform())"
        )
    ],
    capture_output=True,
    text=True,
    timeout=30,
    env=BWGNN_SUBPROCESS_ENV
)

print(
    "\n===== PYTHON 3.9 DETAILS =====",
    flush=True
)

print(
    detail_check.stdout,
    flush=True
)

if detail_check.returncode != 0:

    print(
        detail_check.stderr,
        flush=True
    )

    raise RuntimeError(
        "Isolated Python detail check failed."
    )


# ------------------------------------------------------------
# 10. Final gate
# ------------------------------------------------------------

print(
    "\n===== STEP 6E.2 GATE =====",
    flush=True
)

if (
    python_check.returncode == 0
    and python_version_text.startswith("Python 3.9")
):

    print(
        "PASS — isolated Python 3.9 environment "
        "created successfully with uv.",
        flush=True
    )

else:

    raise RuntimeError(
        "Python 3.9 environment verification failed."
    )


log("DONE — corrected Step 6E.2 complete")

[   0.00s] START — corrected Python 3.9 environment creation

===== ENVIRONMENT CREATION METHOD =====
Tool        : uv
uv path     : /usr/local/bin/uv
Target path : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
GPU policy  : CUDA_VISIBLE_DEVICES=0
[   0.01s] Checking uv version...

===== UV VERSION =====
uv 0.11.13 (x86_64-unknown-linux-gnu)
[   0.09s] Creating isolated Python 3.9 environment with uv...

COMMAND: /usr/local/bin/uv venv --python 3.9 --seed /kaggle/working/comp8851_bwgnn/envs/bwgnn-author
Using CPython 3.9.25
Creating virtual environment with seed packages at: comp8851_bwgnn/envs/bwgnn-author
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.3
 + pip==26.0.1
 + setuptools==82.0.1
 + wheel==0.48.0
Activate with: source comp8851_bwgnn/envs/bwgnn-author/bin/activate

===== CREATED E

### Step 6E.3A — Exact Dependency Wheel Availability Audit

The BWGNN repository specifies:

- PyTorch 1.9.0
- DGL 0.8.1

It does not specify a CUDA build.

Source inspection also shows that the untouched author entry point does not
explicitly move the model, graph or feature tensor to CUDA.

Before selecting the actual installation build, this step verifies the exact
official Python 3.9 Linux wheels available for:

1. PyTorch 1.9.0 + CUDA 11.1
2. DGL 0.8.1 + CUDA 11.1
3. DGL 0.8.1 CPU

The files are downloaded only for provenance and availability verification.
Nothing is installed in this step.

Each downloaded wheel is SHA-256 hashed and retained for the subsequent
environment installation.

In [7]:
# ============================================================
# STEP 6E.3A — EXACT WHEEL AVAILABILITY AUDIT
# ============================================================

import os
import subprocess
import shutil
import hashlib
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - START
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


# ------------------------------------------------------------
# Paths / environment
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

WHEEL_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/wheels"
)

WHEEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV["CUDA_VISIBLE_DEVICES"] = "0"
SUBPROCESS_ENV["DGLBACKEND"] = "pytorch"
SUBPROCESS_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


if not PYTHON39.exists():
    raise RuntimeError(
        "Python 3.9 environment is missing. "
        "Step 6E.2 must pass first."
    )


# ------------------------------------------------------------
# Live command helper
# ------------------------------------------------------------

def run_live(command, timeout=1800):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=SUBPROCESS_ENV
    )

    command_start = time.perf_counter()

    try:

        for line in process.stdout:

            print(line.rstrip(), flush=True)

            elapsed = time.perf_counter() - command_start

            if elapsed > timeout:

                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        return_code = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if return_code != 0:

        raise RuntimeError(
            f"Command failed with return code {return_code}"
        )


# ------------------------------------------------------------
# SHA-256 helper with progress
# ------------------------------------------------------------

def hash_with_progress(path):

    path = Path(path)

    total = path.stat().st_size
    read_bytes = 0

    hasher = hashlib.sha256()

    chunk_size = 8 * 1024 * 1024
    report_every = 256 * 1024 * 1024
    next_report = report_every

    hash_start = time.perf_counter()

    with path.open("rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            hasher.update(chunk)
            read_bytes += len(chunk)

            if (
                read_bytes >= next_report
                or read_bytes == total
            ):

                pct = (
                    read_bytes / total * 100
                    if total
                    else 100
                )

                print(
                    f"  HASH "
                    f"{read_bytes / (1024**2):9.1f} / "
                    f"{total / (1024**2):9.1f} MiB | "
                    f"{pct:6.2f}%",
                    flush=True
                )

                next_report += report_every

    return (
        hasher.hexdigest(),
        time.perf_counter() - hash_start
    )


# ------------------------------------------------------------
# Download helper
# ------------------------------------------------------------

def audit_wheel(
    label,
    requirement,
    find_links
):

    print(
        "\n" + "=" * 100,
        flush=True
    )

    print(
        f"DEPENDENCY: {label}",
        flush=True
    )

    print(
        f"Requirement: {requirement}",
        flush=True
    )

    print(
        "=" * 100,
        flush=True
    )

    target = (
        WHEEL_ROOT /
        label.lower()
        .replace(" ", "_")
        .replace("+", "_")
    )

    if target.exists():
        shutil.rmtree(target)

    target.mkdir(
        parents=True,
        exist_ok=True
    )

    log(f"Checking/downloading {label}...")

    run_live(
        [
            str(PYTHON39),
            "-m",
            "pip",
            "download",
            "--no-deps",
            "--only-binary=:all:",
            "--progress-bar",
            "on",
            "--dest",
            str(target),
            requirement,
            "-f",
            find_links
        ]
    )

    wheels = sorted(
        target.glob("*.whl")
    )

    if len(wheels) != 1:

        raise RuntimeError(
            f"{label}: expected exactly one wheel, "
            f"found {len(wheels)}."
        )

    wheel = wheels[0]

    print(
        "\nDOWNLOADED WHEEL:",
        wheel.name,
        flush=True
    )

    print(
        "SIZE:",
        f"{wheel.stat().st_size:,} bytes "
        f"({wheel.stat().st_size / (1024**2):.2f} MiB)",
        flush=True
    )

    log(f"Hashing {wheel.name}...")

    sha256, hash_seconds = hash_with_progress(
        wheel
    )

    print(
        "SHA-256:",
        sha256,
        flush=True
    )

    print(
        "Hash time:",
        f"{hash_seconds:.2f} s",
        flush=True
    )

    return {
        "label": label,
        "requirement": requirement,
        "wheel": str(wheel),
        "wheel_name": wheel.name,
        "size_bytes": wheel.stat().st_size,
        "sha256": sha256
    }


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — exact wheel availability audit")


# ------------------------------------------------------------
# 1. PyTorch 1.9.0 CUDA 11.1
# ------------------------------------------------------------

torch_record = audit_wheel(
    label="PyTorch_1.9.0_cu111",
    requirement="torch==1.9.0+cu111",
    find_links=(
        "https://download.pytorch.org/"
        "whl/torch_stable.html"
    )
)


# ------------------------------------------------------------
# 2. DGL 0.8.1 CUDA 11.1
# ------------------------------------------------------------

dgl_gpu_record = audit_wheel(
    label="DGL_0.8.1_cu111",
    requirement="dgl-cu111==0.8.1",
    find_links=(
        "https://data.dgl.ai/wheels/repo.html"
    )
)


# ------------------------------------------------------------
# 3. DGL 0.8.1 CPU
# ------------------------------------------------------------

dgl_cpu_record = audit_wheel(
    label="DGL_0.8.1_CPU",
    requirement="dgl==0.8.1",
    find_links=(
        "https://data.dgl.ai/wheels/repo.html"
    )
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

wheel_records = [
    torch_record,
    dgl_gpu_record,
    dgl_cpu_record
]


print(
    "\n" + "=" * 100,
    flush=True
)

print(
    "EXACT WHEEL AVAILABILITY SUMMARY",
    flush=True
)

print(
    "=" * 100,
    flush=True
)


for record in wheel_records:

    print(
        f"\n{record['label']}",
        flush=True
    )

    print(
        f"  wheel  : {record['wheel_name']}",
        flush=True
    )

    print(
        f"  size   : {record['size_bytes']:,} bytes",
        flush=True
    )

    print(
        f"  sha256 : {record['sha256']}",
        flush=True
    )


print(
    "\n===== STEP 6E.3A GATE =====",
    flush=True
)

print(
    "PASS — exact PyTorch and both DGL 0.8.1 "
    "candidate wheels are available for Python 3.9 Linux.",
    flush=True
)

print(
    "\nNO PACKAGE HAS BEEN INSTALLED.",
    flush=True
)

log("DONE — Step 6E.3A complete")

[   0.00s] START — exact wheel availability audit

DEPENDENCY: PyTorch_1.9.0_cu111
Requirement: torch==1.9.0+cu111
[   0.01s] Checking/downloading PyTorch_1.9.0_cu111...

COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip download --no-deps --only-binary=:all: --progress-bar on --dest /kaggle/working/comp8851_bwgnn/wheels/pytorch_1.9.0_cu111 torch==1.9.0+cu111 -f https://download.pytorch.org/whl/torch_stable.html
Looking in links: https://download.pytorch.org/whl/torch_stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 14.1 MB/s  0:00:23
Saved ./comp8851_bwgnn/wheels/pytorch_1.9.0_cu111/torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl
Successfully downloaded torch

DOWNLOADED WHEEL: torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl
SIZE: 2,041,351,091 bytes (1946.78 MiB)
[  35.12s] Hashing torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl...
  HASH     256.0 /    1946.8 MiB |  13.15%
  HASH     512.0 /    1946.8 MiB |  26.30%
  HASH     768.0 /    1946.8 Mi

### Step 6E.3B — Install the Frozen BWGNN Stack

Exact author framework versions selected:

- PyTorch 1.9.0 + CUDA 11.1
- DGL 0.8.1 + CUDA 11.1

The CUDA-enabled DGL build is selected because the controlled benchmark
requires NVIDIA T4 GPU 0. The CPU DGL 0.8.1 wheel downloaded in Step 6E.3A
is retained only as provenance evidence.

The BWGNN repository does not specify versions for NumPy, SciPy,
scikit-learn, NetworkX, SymPy or other supporting packages.

Therefore, compatible Python 3.9-era versions are pinned operationally and
recorded as project compatibility choices, not author requirements.

The exact previously downloaded PyTorch and DGL wheels are installed from
local files so their SHA-256 identities remain fixed.

In [8]:
# ============================================================
# STEP 6E.3B — INSTALL FROZEN BWGNN STACK
# ============================================================

import os
import subprocess
import json
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

WHEEL_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/wheels"
)

EVIDENCE_DIR = Path(
    "/kaggle/working/comp8851_bwgnn/evidence"
)

EVIDENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TORCH_WHEEL = (
    WHEEL_ROOT /
    "pytorch_1.9.0_cu111" /
    "torch-1.9.0+cu111-cp39-cp39-linux_x86_64.whl"
)

DGL_WHEEL = (
    WHEEL_ROOT /
    "dgl_0.8.1_cu111" /
    "dgl_cu111-0.8.1-cp39-cp39-manylinux1_x86_64.whl"
)


for path in [
    PYTHON39,
    TORCH_WHEEL,
    DGL_WHEEL
]:
    if not path.exists():
        raise FileNotFoundError(path)


# ------------------------------------------------------------
# Controlled subprocess environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


# ------------------------------------------------------------
# Live subprocess helper
# ------------------------------------------------------------

def run_live(command, timeout=1200):

    print(
        "\nCOMMAND:",
        " ".join(map(str, command)),
        flush=True
    )

    process = subprocess.Popen(
        list(map(str, command)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=ENV
    )

    started = time.perf_counter()

    try:

        for line in process.stdout:

            print(
                line.rstrip(),
                flush=True
            )

            if (
                time.perf_counter() - started
                > timeout
            ):
                process.kill()

                raise TimeoutError(
                    f"Command exceeded {timeout} seconds."
                )

        rc = process.wait()

    finally:

        if process.stdout:
            process.stdout.close()

    if rc != 0:
        raise RuntimeError(
            f"Command failed with return code {rc}"
        )


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — frozen BWGNN stack installation")


# ------------------------------------------------------------
# 1. Compatibility support packages
#
# These are PROJECT compatibility pins.
# They are NOT claimed as author-specified versions.
# ------------------------------------------------------------

SUPPORT_PACKAGES = [

    "numpy==1.23.5",
    "scipy==1.9.3",

    "scikit-learn==1.1.3",

    "networkx==2.8.8",

    "sympy==1.10.1",

    "requests==2.28.1",
    "tqdm==4.64.1",
    "psutil==5.9.4",

    "typing_extensions==4.4.0",
]


print(
    "\n===== OPERATIONAL COMPATIBILITY PINS =====",
    flush=True
)

for package in SUPPORT_PACKAGES:
    print(
        package,
        flush=True
    )


log("Installing compatibility support packages...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "on",
        *SUPPORT_PACKAGES
    ],
    timeout=900
)


# ------------------------------------------------------------
# 2. Install exact PyTorch wheel
# ------------------------------------------------------------

log("Installing frozen PyTorch 1.9.0+cu111 wheel...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--no-deps",
        str(TORCH_WHEEL)
    ],
    timeout=1200
)


# ------------------------------------------------------------
# 3. Install exact DGL wheel
# ------------------------------------------------------------

log("Installing frozen DGL 0.8.1 cu111 wheel...")

run_live(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "install",
        "--no-deps",
        str(DGL_WHEEL)
    ],
    timeout=900
)


# ------------------------------------------------------------
# 4. pip dependency consistency check
# ------------------------------------------------------------

log("Running pip dependency check...")

pip_check = subprocess.run(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "check"
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=ENV
)


print(
    "\n===== PIP CHECK =====",
    flush=True
)

print(
    pip_check.stdout.strip()
    or pip_check.stderr.strip(),
    flush=True
)


if pip_check.returncode != 0:

    raise RuntimeError(
        "pip dependency consistency check failed."
    )


# ------------------------------------------------------------
# 5. Save selection record
# ------------------------------------------------------------

selection = {

    "author_requested": {
        "torch": "1.9.0",
        "dgl": "0.8.1",
        "python": "NOT SPECIFIED"
    },

    "actual_selected": {
        "python": "3.9.25",
        "torch": "1.9.0+cu111",
        "dgl": "0.8.1 cu111"
    },

    "framework_wheels": {

        "torch": {
            "filename": TORCH_WHEEL.name,
            "sha256":
                "5422d19042e217c2aa94030b16b3fe4da5be9ba8eea46e7e59d40a110955962d"
        },

        "dgl": {
            "filename": DGL_WHEEL.name,
            "sha256":
                "48152794ea2744196e0f7d70d257d3a8f565382855f25efa4edef08e9f9f0b87"
        }
    },

    "operational_compatibility_pins":
        SUPPORT_PACKAGES,

    "gpu_policy":
        "CUDA_VISIBLE_DEVICES=0; NVIDIA Tesla T4"
}


selection_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_selection.json"
)

selection_path.write_text(
    json.dumps(
        selection,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "\n===== SAVED ENVIRONMENT SELECTION =====",
    flush=True
)

print(
    selection_path,
    flush=True
)


print(
    "\n===== STEP 6E.3B GATE =====",
    flush=True
)

print(
    "PASS — exact PyTorch/DGL wheels installed "
    "and pip dependency check passed.",
    flush=True
)

log("DONE — Step 6E.3B complete")

[   0.00s] START — frozen BWGNN stack installation

===== OPERATIONAL COMPATIBILITY PINS =====
numpy==1.23.5
scipy==1.9.3
scikit-learn==1.1.3
networkx==2.8.8
sympy==1.10.1
requests==2.28.1
tqdm==4.64.1
psutil==5.9.4
typing_extensions==4.4.0
[   0.01s] Installing compatibility support packages...

COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip install --progress-bar on numpy==1.23.5 scipy==1.9.3 scikit-learn==1.1.3 networkx==2.8.8 sympy==1.10.1 requests==2.28.1 tqdm==4.64.1 psutil==5.9.4 typing_extensions==4.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 13.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 21.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 28.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 11.5 MB/s  0:00:00


### Step 6E.4 — Verify the Installed BWGNN Environment

The isolated environment is tested in a fresh Python 3.9 subprocess with
GPU visibility explicitly restricted to GPU 0.

The gate verifies:

- Python 3.9
- PyTorch 1.9.0
- PyTorch CUDA 11.1 build
- DGL 0.8.1
- supporting dependency versions
- CUDA availability
- exactly one visible GPU
- Tesla T4 identity
- CUDA tensor execution
- basic DGL graph creation

The Kaggle base Python 3.12 environment is not used for BWGNN execution.

In [9]:
# ============================================================
# STEP 6E.4 — VERIFY INSTALLED BWGNN ENVIRONMENT
# ============================================================

import os
import subprocess
import json
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"


log("START — isolated BWGNN environment verification")


VERIFY_SCRIPT = r'''
import json
import sys

result = {}

result["python"] = sys.version

import numpy
import scipy
import sklearn
import networkx
import sympy
import requests
import tqdm
import psutil

import torch

# Import torch BEFORE DGL so CUDA runtime libraries are loaded.
import dgl


result["numpy"] = numpy.__version__
result["scipy"] = scipy.__version__
result["sklearn"] = sklearn.__version__
result["networkx"] = networkx.__version__
result["sympy"] = sympy.__version__
result["requests"] = requests.__version__
result["tqdm"] = tqdm.__version__
result["psutil"] = psutil.__version__

result["torch"] = torch.__version__
result["torch_cuda_build"] = torch.version.cuda

result["dgl"] = dgl.__version__

result["cuda_available"] = torch.cuda.is_available()
result["visible_gpu_count"] = torch.cuda.device_count()


if torch.cuda.is_available():

    result["gpu0"] = torch.cuda.get_device_name(0)

    x = torch.tensor(
        [1.0, 2.0, 3.0],
        device="cuda:0"
    )

    result["cuda_tensor_sum"] = float(
        x.sum().item()
    )

else:

    result["gpu0"] = None
    result["cuda_tensor_sum"] = None


# Basic DGL CPU graph creation sanity test.
g = dgl.graph(
    ([0, 1], [1, 2])
)

result["dgl_test_nodes"] = g.num_nodes()
result["dgl_test_edges"] = g.num_edges()


print(
    json.dumps(result)
)
'''


log("Launching fresh Python 3.9 subprocess...")

verify = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        VERIFY_SCRIPT
    ],
    capture_output=True,
    text=True,
    timeout=180,
    env=ENV
)


print(
    "\n===== VERIFICATION STDERR =====",
    flush=True
)

print(
    verify.stderr.strip()
    if verify.stderr.strip()
    else "(none)",
    flush=True
)


if verify.returncode != 0:

    print(
        "\n===== VERIFICATION STDOUT =====",
        flush=True
    )

    print(
        verify.stdout,
        flush=True
    )

    raise RuntimeError(
        "Isolated BWGNN environment verification failed."
    )


raw_lines = [
    line
    for line in verify.stdout.splitlines()
    if line.strip()
]

data = json.loads(
    raw_lines[-1]
)


print(
    "\n===== BWGNN ISOLATED ENVIRONMENT =====",
    flush=True
)

for key, value in data.items():

    print(
        f"{key:<22}: {value}",
        flush=True
    )


# ------------------------------------------------------------
# Required gates
# ------------------------------------------------------------

checks = {

    "Python 3.9":
        data["python"].startswith("3.9"),

    "PyTorch 1.9.0":
        data["torch"].startswith("1.9.0"),

    "PyTorch CUDA build 11.1":
        str(data["torch_cuda_build"]).startswith("11.1"),

    "DGL 0.8.1":
        data["dgl"].startswith("0.8.1"),

    "CUDA available":
        data["cuda_available"] is True,

    "Exactly one visible GPU":
        data["visible_gpu_count"] == 1,

    "GPU 0 is Tesla T4":
        "T4" in (data["gpu0"] or "").upper(),

    "CUDA tensor executes":
        data["cuda_tensor_sum"] == 6.0,

    "DGL graph creation":
        (
            data["dgl_test_nodes"] == 3
            and
            data["dgl_test_edges"] == 2
        )
}


print(
    "\n===== ENVIRONMENT GATE CHECKS =====",
    flush=True
)

for name, passed in checks.items():

    print(
        f"{'PASS' if passed else 'FAIL'} | {name}",
        flush=True
    )


if not all(checks.values()):

    raise RuntimeError(
        "BWGNN environment gate failed."
    )


# ------------------------------------------------------------
# Freeze complete environment
# ------------------------------------------------------------

log("Capturing complete package freeze...")

freeze = subprocess.run(
    [
        str(PYTHON39),
        "-m",
        "pip",
        "freeze"
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=ENV,
    check=True
)


freeze_path = (
    EVIDENCE_DIR /
    "bwgnn_environment_freeze.txt"
)

freeze_path.write_text(
    freeze.stdout,
    encoding="utf-8"
)


print(
    "\n===== SAVED ENVIRONMENT FREEZE =====",
    flush=True
)

print(
    freeze_path,
    flush=True
)


print(
    "\n===== STEP 6E GATE =====",
    flush=True
)

print(
    "PASS — isolated BWGNN environment is operational "
    "with PyTorch 1.9.0, DGL 0.8.1 and Tesla T4 GPU 0.",
    flush=True
)

log("DONE — Step 6E complete")

[   0.00s] START — isolated BWGNN environment verification
[   0.00s] Launching fresh Python 3.9 subprocess...

===== VERIFICATION STDERR =====
Traceback (most recent call last):
  File "<string>", line 21, in <module>
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/__init__.py", line 16, in <module>
    from .backend import load_backend, backend_name
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/backend/__init__.py", line 109, in <module>
    load_backend(get_preferred_backend())
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/backend/__init__.py", line 43, in load_backend
    from .._ffi.base import load_tensor_adapter # imports DGL C library
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/_ffi/base.py", line 45, in <module>
    _LIB, _LIB_NAME, _DIR_NAME = _load_lib()
  File "/kaggle/working/comp8851_bwgnn/envs/bwgnn-au

RuntimeError: Isolated BWGNN environment verification failed.

### Step 6E.4A — CUDA Runtime Library Linkage Audit

The first environment verification successfully reached the installed DGL
package but DGL's native library could not locate `libcublas.so.11`.

This indicates a CUDA runtime library search-path issue rather than an
immediate package-version mismatch.

Before changing any package, this step checks:

- whether `libcublas.so.11` already exists inside the isolated environment
- whether it is bundled with the PyTorch 1.9.0 CUDA wheel
- the location of DGL's native shared library
- the unresolved native dependencies reported by `ldd`
- the current library search path

No package is installed, removed or modified in this step.

In [10]:
# ============================================================
# STEP 6E.4A — CUDA LIBRARY LINKAGE AUDIT
# ============================================================

import os
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE_PACKAGES = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

TORCH_DIR = SITE_PACKAGES / "torch"
DGL_DIR = SITE_PACKAGES / "dgl"


log("START — CUDA runtime library linkage audit")


# ------------------------------------------------------------
# 1. Current relevant environment variables
# ------------------------------------------------------------

print(
    "\n===== CURRENT LIBRARY ENVIRONMENT =====",
    flush=True
)

print(
    "LD_LIBRARY_PATH       :",
    os.environ.get("LD_LIBRARY_PATH"),
    flush=True
)

print(
    "CUDA_VISIBLE_DEVICES  :",
    os.environ.get("CUDA_VISIBLE_DEVICES"),
    flush=True
)

print(
    "Torch directory       :",
    TORCH_DIR,
    flush=True
)

print(
    "DGL directory         :",
    DGL_DIR,
    flush=True
)


# ------------------------------------------------------------
# 2. Search isolated environment for CUDA libraries
# ------------------------------------------------------------

log("Searching isolated environment for libcublas...")

cublas_files = sorted(
    BWGNN_ENV.rglob("libcublas.so*")
)

print(
    "\n===== LIBCUBLAS FILES IN ISOLATED ENVIRONMENT =====",
    flush=True
)

if cublas_files:

    for path in cublas_files:
        print(path, flush=True)

else:

    print(
        "NONE FOUND",
        flush=True
    )


# ------------------------------------------------------------
# 3. Search specifically inside torch/lib
# ------------------------------------------------------------

TORCH_LIB = TORCH_DIR / "lib"

print(
    "\n===== TORCH LIB DIRECTORY =====",
    flush=True
)

print(
    "Path   :",
    TORCH_LIB,
    flush=True
)

print(
    "Exists :",
    TORCH_LIB.exists(),
    flush=True
)


if TORCH_LIB.exists():

    torch_cuda_libs = sorted(
        [
            p
            for p in TORCH_LIB.iterdir()
            if (
                "cublas" in p.name.lower()
                or "cudart" in p.name.lower()
                or "cudnn" in p.name.lower()
                or "cusparse" in p.name.lower()
                or "curand" in p.name.lower()
            )
        ],
        key=lambda p: p.name
    )

    print(
        "\nRelevant CUDA libraries in torch/lib:",
        flush=True
    )

    if torch_cuda_libs:

        for path in torch_cuda_libs:
            print(
                f"  {path.name}",
                flush=True
            )

    else:

        print(
            "  NONE FOUND",
            flush=True
        )


# ------------------------------------------------------------
# 4. Locate DGL native libraries
# ------------------------------------------------------------

log("Locating DGL native shared libraries...")

dgl_shared = sorted(
    DGL_DIR.rglob("*.so")
)

print(
    "\n===== DGL SHARED LIBRARIES =====",
    flush=True
)

for path in dgl_shared:

    print(
        path,
        flush=True
    )


# Prefer libdgl.so for dependency inspection
libdgl_candidates = [
    p
    for p in dgl_shared
    if p.name == "libdgl.so"
]

if not libdgl_candidates:

    raise RuntimeError(
        "Could not locate DGL libdgl.so."
    )

LIBDGL = libdgl_candidates[0]


# ------------------------------------------------------------
# 5. Run ldd on DGL native library
# ------------------------------------------------------------

log("Running ldd on DGL native library...")

ldd = subprocess.run(
    [
        "ldd",
        str(LIBDGL)
    ],
    capture_output=True,
    text=True,
    timeout=60
)


print(
    "\n===== LDD — DGL NATIVE LIBRARY =====",
    flush=True
)

print(
    "Library:",
    LIBDGL,
    flush=True
)

print(
    ldd.stdout,
    flush=True
)

if ldd.stderr:
    print(
        ldd.stderr,
        flush=True
    )


# ------------------------------------------------------------
# 6. Extract missing libraries
# ------------------------------------------------------------

missing_lines = [
    line.strip()
    for line in ldd.stdout.splitlines()
    if "not found" in line.lower()
]


print(
    "\n===== UNRESOLVED DGL DEPENDENCIES =====",
    flush=True
)

if missing_lines:

    for line in missing_lines:
        print(
            line,
            flush=True
        )

else:

    print(
        "NONE",
        flush=True
    )


# ------------------------------------------------------------
# 7. Check PyTorch itself in a clean subprocess
# ------------------------------------------------------------

log("Checking PyTorch CUDA metadata without importing DGL...")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

torch_check = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        (
            "import torch; "
            "print('torch=', torch.__version__); "
            "print('torch_cuda=', torch.version.cuda); "
            "print('cuda_available=', torch.cuda.is_available()); "
            "print('gpu_count=', torch.cuda.device_count()); "
            "print('gpu0=', torch.cuda.get_device_name(0) "
            "if torch.cuda.is_available() else None)"
        )
    ],
    capture_output=True,
    text=True,
    timeout=120,
    env=env
)


print(
    "\n===== PYTORCH-ONLY CUDA CHECK =====",
    flush=True
)

print(
    torch_check.stdout,
    flush=True
)

if torch_check.stderr:

    print(
        "STDERR:",
        torch_check.stderr,
        flush=True
    )


print(
    "\n===== STEP 6E.4A STATUS =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no packages or libraries were changed.",
    flush=True
)

log("DONE — Step 6E.4A complete")

[   0.00s] START — CUDA runtime library linkage audit

===== CURRENT LIBRARY ENVIRONMENT =====
LD_LIBRARY_PATH       : /usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64
CUDA_VISIBLE_DEVICES  : None
Torch directory       : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch
DGL directory         : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl
[   0.01s] Searching isolated environment for libcublas...

===== LIBCUBLAS FILES IN ISOLATED ENVIRONMENT =====
NONE FOUND

===== TORCH LIB DIRECTORY =====
Path   : /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/torch/lib
Exists : True

Relevant CUDA libraries in torch/lib:
  libcudart-6d56b25a.so.11.0
[   0.11s] Locating DGL native shared libraries...

===== DGL SHARED LIBRARIES =====
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/dgl/_ffi/_cy3/core.cpython-39-x86_64-linux-gnu.so
/kaggle/working/comp8851_b

### Step 6E.4B — NVIDIA CUDA 11 Runtime Availability Audit

The DGL 0.8.1 CUDA 11.1 wheel imports successfully only if its native
dependencies can be resolved.

The previous linkage audit identified three unresolved libraries:

- `libcudart.so.11.0`
- `libcublas.so.11`
- `libcusparse.so.11`

PyTorch 1.9.0+cu111 itself is operational on the Tesla T4, so the framework
installation is retained unchanged.

This step queries NVIDIA's package index for available CUDA 11 runtime,
cuBLAS and cuSPARSE packages before any additional runtime library is
installed.

No package is modified in this step.

In [11]:
# ============================================================
# STEP 6E.4B — NVIDIA CUDA-11 RUNTIME AVAILABILITY AUDIT
# ============================================================

import os
import subprocess
import time
from pathlib import Path

START = time.perf_counter()

def log(message):
    print(
        f"[{time.perf_counter() - START:7.2f}s] {message}",
        flush=True
    )


BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

if not PYTHON39.exists():
    raise RuntimeError(
        "Isolated Python 3.9 environment is missing."
    )


ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"


NVIDIA_INDEX = "https://pypi.nvidia.com"


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def query_versions(package):

    print(
        "\n" + "=" * 90,
        flush=True
    )

    print(
        f"PACKAGE: {package}",
        flush=True
    )

    print(
        "=" * 90,
        flush=True
    )

    command = [
        str(PYTHON39),
        "-m",
        "pip",
        "index",
        "versions",
        package,
        "--index-url",
        NVIDIA_INDEX
    ]

    print(
        "COMMAND:",
        " ".join(command),
        flush=True
    )

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=120,
        env=ENV
    )

    print(
        "\nSTDOUT:",
        flush=True
    )

    print(
        result.stdout.strip()
        if result.stdout.strip()
        else "(none)",
        flush=True
    )

    print(
        "\nSTDERR:",
        flush=True
    )

    print(
        result.stderr.strip()
        if result.stderr.strip()
        else "(none)",
        flush=True
    )

    return {
        "package": package,
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr
    }


# ------------------------------------------------------------
# Start
# ------------------------------------------------------------

log("START — NVIDIA CUDA-11 runtime availability audit")


packages = [
    "nvidia-cuda-runtime",
    "nvidia-cublas",
    "nvidia-cusparse",
    "nvidia-cuda-runtime-cu11",
    "nvidia-cublas-cu11",
    "nvidia-cusparse-cu11",
]


records = []

for package in packages:

    log(
        f"Querying {package}..."
    )

    records.append(
        query_versions(package)
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "\n" + "=" * 90,
    flush=True
)

print(
    "QUERY SUMMARY",
    flush=True
)

print(
    "=" * 90,
    flush=True
)


for record in records:

    status = (
        "QUERY OK"
        if record["returncode"] == 0
        else "NO RESULT / QUERY FAILED"
    )

    print(
        f"{record['package']:<28} : {status}",
        flush=True
    )


print(
    "\n===== STEP 6E.4B STATUS =====",
    flush=True
)

print(
    "AUDIT COMPLETE — no CUDA runtime package was installed.",
    flush=True
)

log("DONE — Step 6E.4B complete")

[   0.00s] START — NVIDIA CUDA-11 runtime availability audit
[   0.00s] Querying nvidia-cuda-runtime...

PACKAGE: nvidia-cuda-runtime
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip index versions nvidia-cuda-runtime --index-url https://pypi.nvidia.com

STDOUT:
nvidia-cuda-runtime (13.3.29)
Available versions: 13.3.29, 13.2.86, 13.2.75, 13.2.51, 13.1.80, 13.0.96, 13.0.88, 13.0.48, 11.3.58, 11.2.146, 11.2.72, 11.1.74

STDERR:
(none)
[   0.66s] Querying nvidia-cublas...

PACKAGE: nvidia-cublas
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python -m pip index versions nvidia-cublas --index-url https://pypi.nvidia.com

STDOUT:
nvidia-cublas (13.6.1.10)
Available versions: 13.6.1.10, 13.6.0.2, 13.5.1.27, 13.4.1.3, 13.4.1.1, 13.4.0.1, 13.3.0.5, 13.2.2.2, 13.2.1.1, 13.2.0.9, 13.1.1.3, 13.1.0.3, 13.0.2.14, 13.0.0.19

STDERR:
(none)
[   1.34s] Querying nvidia-cusparse...

PACKAGE: nvidia-cusparse
COMMAND: /kaggle/working/comp8851_bwgnn/envs/bwgnn-auth

### Step 6E.4C — Fix DGL CUDA Runtime Linkage

DGL 0.8.1 requires three CUDA 11 shared libraries that are missing from the
current library search path:

- libcudart.so.11.0
- libcublas.so.11
- libcusparse.so.11

CUDA 11 runtime packages are added without changing PyTorch 1.9.0 or
DGL 0.8.1. The environment is then retested.

In [12]:
# ============================================================
# STEP 6E.4C — INSTALL MISSING CUDA 11 LIBRARIES + RETEST DGL
# ============================================================

import os
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

print("START — installing missing CUDA 11 runtime libraries", flush=True)

# ------------------------------------------------------------
# 1. Install only the required CUDA 11 runtime libraries
# ------------------------------------------------------------

packages = [
    "nvidia-cuda-runtime-cu11==11.8.89",
    "nvidia-cublas-cu11==11.11.3.6",
    "nvidia-cusparse-cu11==11.7.5.86",
]

cmd = [
    str(PYTHON39),
    "-m",
    "pip",
    "install",
    "--no-deps",
    *packages
]

print("\nInstalling:", flush=True)

for p in packages:
    print(" ", p, flush=True)

result = subprocess.run(
    cmd,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        "CUDA 11 runtime library installation failed."
    )


# ------------------------------------------------------------
# 2. Find installed NVIDIA library directories
# ------------------------------------------------------------

lib_dirs = []

for pattern in [
    "nvidia/cuda_runtime/lib",
    "nvidia/cublas/lib",
    "nvidia/cusparse/lib",
]:

    path = SITE / pattern

    if path.exists():
        lib_dirs.append(str(path))


print("\n===== CUDA LIBRARY DIRECTORIES =====", flush=True)

for path in lib_dirs:
    print(path, flush=True)


if len(lib_dirs) != 3:
    raise RuntimeError(
        "Could not locate all three NVIDIA CUDA library directories."
    )


# ------------------------------------------------------------
# 3. Build clean BWGNN subprocess environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

existing_ld = ENV.get(
    "LD_LIBRARY_PATH",
    ""
)

ENV["LD_LIBRARY_PATH"] = (
    ":".join(lib_dirs)
    + (":" + existing_ld if existing_ld else "")
)


# ------------------------------------------------------------
# 4. Re-run ldd
# ------------------------------------------------------------

LIBDGL = SITE / "dgl/libdgl.so"

ldd = subprocess.run(
    ["ldd", str(LIBDGL)],
    capture_output=True,
    text=True,
    env=ENV
)

missing = [
    line.strip()
    for line in ldd.stdout.splitlines()
    if "not found" in line.lower()
]


print("\n===== UNRESOLVED DGL DEPENDENCIES =====", flush=True)

if missing:
    for line in missing:
        print(line, flush=True)
else:
    print("NONE", flush=True)


if missing:
    raise RuntimeError(
        "DGL still has unresolved shared-library dependencies."
    )


# ------------------------------------------------------------
# 5. Fresh Torch + DGL verification
# ------------------------------------------------------------

verify_code = r'''
import torch
import dgl

print("torch =", torch.__version__)
print("torch CUDA =", torch.version.cuda)
print("dgl =", dgl.__version__)
print("CUDA available =", torch.cuda.is_available())
print("visible GPUs =", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU 0 =", torch.cuda.get_device_name(0))

x = torch.tensor([1., 2., 3.], device="cuda:0")
print("CUDA tensor sum =", x.sum().item())

g = dgl.graph(([0, 1], [1, 2]))
print("DGL nodes =", g.num_nodes())
print("DGL edges =", g.num_edges())
'''

verify = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        verify_code
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=180
)


print("\n===== TORCH + DGL VERIFICATION =====", flush=True)

print(verify.stdout, flush=True)

if verify.stderr:
    print("STDERR:", flush=True)
    print(verify.stderr, flush=True)


if verify.returncode != 0:
    raise RuntimeError(
        "Torch/DGL verification failed."
    )


print("===== STEP 6E.4C GATE =====", flush=True)

print(
    "PASS — DGL 0.8.1 loads successfully with CUDA 11 runtime libraries.",
    flush=True
)

START — installing missing CUDA 11 runtime libraries

Installing:
  nvidia-cuda-runtime-cu11==11.8.89
  nvidia-cublas-cu11==11.11.3.6
  nvidia-cusparse-cu11==11.7.5.86
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 61.0 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 76.6 MB/s  0:00:02


===== CUDA LIBRARY DIRECTORIES =====
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cuda_runtime/lib
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cublas/lib
/kaggle/working/comp8851_bwgnn/envs/bwgnn-author/lib/python3.9/site-packages/nvidia/cusparse/lib

===== UNRESOLVED DGL DEPENDENCIES =====
NONE

===== TORCH + DGL VERIFICATION =====
torch = 1.9.0+cu111
torch CUDA = 11.1
dgl = 0.8.1
CUDA available = True
visible GPUs = 1
GPU 0 = Tesla T4
CUDA tensor sum = 6.0
DGL nodes = 3
DGL edges = 2

===== STEP 6E.4C GATE ===

## Step 7 — Canonical Dataset Inputs

The benchmark datasets are attached from the same Kaggle dataset sources
used by the dataset registry notebook.

This step verifies that the expected canonical input files are available
before BWGNN loads any graph.

In [13]:
# ============================================================
# STEP 7A — VERIFY CANONICAL DATASET INPUT PATHS
# ============================================================

from pathlib import Path

DATASETS = {
    "YelpChi":
        Path("/kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat"),

    "Amazon":
        Path("/kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat"),

    "T-Finance":
        Path("/kaggle/input/datasets/pathikahmed0007/tfinance/tfinance"),

    "T-Social":
        Path("/kaggle/input/datasets/pathikahmed0007/tsocial/tsocial"),

    "Elliptic Features":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv"),

    "Elliptic Classes":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv"),

    "Elliptic Edges":
        Path("/kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv"),

    "FDCompCN":
        Path("/kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl"),
}


print("===== CANONICAL DATASET INPUT CHECK =====")

all_ok = True

for name, path in DATASETS.items():

    exists = path.exists()

    print(
        f"{'PASS' if exists else 'FAIL'} | "
        f"{name:<18} | {path}"
    )

    if not exists:
        all_ok = False


print("\n===== STEP 7A GATE =====")

if all_ok:
    print("PASS — all canonical dataset inputs are attached.")
else:
    raise RuntimeError(
        "One or more canonical dataset inputs are missing."
    )

===== CANONICAL DATASET INPUT CHECK =====
PASS | YelpChi            | /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
PASS | Amazon             | /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat
PASS | T-Finance          | /kaggle/input/datasets/pathikahmed0007/tfinance/tfinance
PASS | T-Social           | /kaggle/input/datasets/pathikahmed0007/tsocial/tsocial
PASS | Elliptic Features  | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv
PASS | Elliptic Classes   | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv
PASS | Elliptic Edges     | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv
PASS | FDCompCN           | /kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl

===== STEP 7A GATE =====
PASS — all canonical dataset inputs are attached.


### Step 7B — T-Finance and T-Social Structural Verification

T-Finance and T-Social are loaded with the verified BWGNN-compatible
DGL 0.8.1 environment.

For each dataset this step records:

- number of nodes and edges
- node-data fields
- feature shape
- label shape
- class counts

T-Finance is additionally compared against the previously established
structural target because its current source-file checksum differs from the
earlier audited mirror.

In [14]:
# ============================================================
# STEP 7B — T-FINANCE / T-SOCIAL DGL STRUCTURAL VERIFICATION
# ============================================================

import os
import json
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

DATASETS = {
    "T-Finance":
        "/kaggle/input/datasets/pathikahmed0007/tfinance/tfinance",

    "T-Social":
        "/kaggle/input/datasets/pathikahmed0007/tsocial/tsocial",
}


# ------------------------------------------------------------
# DGL runtime environment
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)


# ------------------------------------------------------------
# Separate subprocess = memory released after each dataset
# ------------------------------------------------------------

CHECK_SCRIPT = r'''
import sys
import json
import torch
import dgl
from dgl.data.utils import load_graphs

path = sys.argv[1]
name = sys.argv[2]

print(f"START | loading {name}", flush=True)

graphs, label_dict = load_graphs(path)

if len(graphs) == 0:
    raise RuntimeError("No graph found in DGL file.")

g = graphs[0]

ndata_keys = list(g.ndata.keys())
edata_keys = list(g.edata.keys())

# Feature field
feature = None
feature_key = None

for candidate in ["feature", "feat", "features"]:
    if candidate in g.ndata:
        feature_key = candidate
        feature = g.ndata[candidate]
        break

# Label field
label = None
label_source = None

if "label" in label_dict:
    label = label_dict["label"]
    label_source = "label_dict['label']"

elif "label" in g.ndata:
    label = g.ndata["label"]
    label_source = "graph.ndata['label']"


summary = {
    "dataset": name,
    "dgl_version": dgl.__version__,
    "torch_version": torch.__version__,
    "num_graphs": len(graphs),
    "num_nodes": int(g.num_nodes()),
    "num_edges": int(g.num_edges()),
    "ndata_keys": ndata_keys,
    "edata_keys": edata_keys,
    "feature_key": feature_key,
    "feature_shape":
        list(feature.shape) if feature is not None else None,
    "label_source": label_source,
    "label_shape":
        list(label.shape) if label is not None else None,
}


if label is not None:

    flat = label.reshape(-1).long()

    values, counts = torch.unique(
        flat,
        return_counts=True
    )

    summary["label_counts"] = {
        str(int(v)): int(c)
        for v, c in zip(values, counts)
    }

else:

    summary["label_counts"] = None


print("RESULT_JSON=" + json.dumps(summary), flush=True)
'''


summaries = {}

for name, path in DATASETS.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    result = subprocess.run(
        [
            str(PYTHON39),
            "-c",
            CHECK_SCRIPT,
            path,
            name
        ],
        capture_output=True,
        text=True,
        env=ENV,
        timeout=900
    )

    print(result.stdout, flush=True)

    if result.stderr:
        print("STDERR:", flush=True)
        print(result.stderr, flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed to load."
        )

    json_line = [
        line
        for line in result.stdout.splitlines()
        if line.startswith("RESULT_JSON=")
    ][0]

    summaries[name] = json.loads(
        json_line.replace(
            "RESULT_JSON=",
            "",
            1
        )
    )


# ------------------------------------------------------------
# Human-readable summary
# ------------------------------------------------------------

print("\n===== STRUCTURAL SUMMARY =====")

for name, s in summaries.items():

    print(f"\n{name}")
    print("  DGL         :", s["dgl_version"])
    print("  Nodes       :", f"{s['num_nodes']:,}")
    print("  Edges       :", f"{s['num_edges']:,}")
    print("  Node fields :", s["ndata_keys"])
    print("  Feature key :", s["feature_key"])
    print("  Feature     :", s["feature_shape"])
    print("  Label source:", s["label_source"])
    print("  Label shape :", s["label_shape"])
    print("  Label counts:", s["label_counts"])


# ------------------------------------------------------------
# T-Finance known structural target
# ------------------------------------------------------------

tf = summaries["T-Finance"]

tf_checks = {
    "39,357 nodes":
        tf["num_nodes"] == 39357,

    "42,445,086 stored edges":
        tf["num_edges"] == 42445086,

    "10 feature dimensions":
        (
            tf["feature_shape"] is not None
            and len(tf["feature_shape"]) == 2
            and tf["feature_shape"][1] == 10
        ),

    "37,554 normal labels":
        tf["label_counts"] is not None
        and tf["label_counts"].get("0") == 37554,

    "1,803 fraud labels":
        tf["label_counts"] is not None
        and tf["label_counts"].get("1") == 1803,
}


print("\n===== T-FINANCE STRUCTURAL TARGET =====")

for check, passed in tf_checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} | {check}"
    )


print("\n===== STEP 7B GATE =====")

if (
    all(tf_checks.values())
    and summaries["T-Social"]["feature_shape"] is not None
    and summaries["T-Social"]["label_counts"] is not None
):

    print(
        "PASS — T-Finance matches the established structural target "
        "and T-Social loads successfully with features and labels."
    )

else:

    raise RuntimeError(
        "Dataset structural verification requires review."
    )


T-Finance
START | loading T-Finance
RESULT_JSON={"dataset": "T-Finance", "dgl_version": "0.8.1", "torch_version": "1.9.0+cu111", "num_graphs": 1, "num_nodes": 39357, "num_edges": 42445086, "ndata_keys": ["label", "feature"], "edata_keys": [], "feature_key": "feature", "feature_shape": [39357, 10], "label_source": "graph.ndata['label']", "label_shape": [39357, 2], "label_counts": {"0": 39357, "1": 39357}}


T-Social
START | loading T-Social
RESULT_JSON={"dataset": "T-Social", "dgl_version": "0.8.1", "torch_version": "1.9.0+cu111", "num_graphs": 1, "num_nodes": 5781065, "num_edges": 146211016, "ndata_keys": ["feature", "label", "_ID"], "edata_keys": ["_ID"], "feature_key": "feature", "feature_shape": [5781065, 10], "label_source": "graph.ndata['label']", "label_shape": [5781065], "label_counts": {"0": 5606785, "1": 174280}}


===== STRUCTURAL SUMMARY =====

T-Finance
  DGL         : 0.8.1
  Nodes       : 39,357
  Edges       : 42,445,086
  Node fields : ['label', 'feature']
  Feature ke

RuntimeError: Dataset structural verification requires review.

### Step 7B Correction — T-Finance Label Encoding

T-Finance stores labels as a two-column one-hot tensor rather than a
one-dimensional class vector.

The previous verification flattened this tensor, which incorrectly counted
both entries of every row.

This correction converts the one-hot labels to class IDs using `argmax`
before calculating the class distribution.

In [15]:
# ============================================================
# STEP 7B CORRECTION — T-FINANCE LABEL COUNTS
# ============================================================

import subprocess
import json

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

CHECK = r'''
import sys
import json
import torch
from dgl.data.utils import load_graphs

graphs, _ = load_graphs(sys.argv[1])
g = graphs[0]

labels = g.ndata["label"]

print("Original label shape:", list(labels.shape))

# T-Finance uses one-hot labels [N, 2]
if labels.ndim == 2 and labels.shape[1] > 1:
    class_ids = torch.argmax(labels, dim=1)
    encoding = "one-hot -> argmax"
else:
    class_ids = labels.reshape(-1).long()
    encoding = "1D class labels"

values, counts = torch.unique(
    class_ids,
    return_counts=True
)

label_counts = {
    str(int(v)): int(c)
    for v, c in zip(values, counts)
}

result = {
    "nodes": int(g.num_nodes()),
    "edges": int(g.num_edges()),
    "features": list(g.ndata["feature"].shape),
    "label_shape": list(labels.shape),
    "label_encoding": encoding,
    "label_counts": label_counts
}

print("RESULT_JSON=" + json.dumps(result))
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        CHECK,
        TF_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "T-Finance corrected verification failed."
    )

line = [
    x for x in result.stdout.splitlines()
    if x.startswith("RESULT_JSON=")
][0]

tf = json.loads(
    line.replace("RESULT_JSON=", "", 1)
)

print("===== CORRECTED T-FINANCE SUMMARY =====")
print("Nodes        :", f"{tf['nodes']:,}")
print("Edges        :", f"{tf['edges']:,}")
print("Features     :", tf["features"])
print("Label shape  :", tf["label_shape"])
print("Encoding     :", tf["label_encoding"])
print("Label counts :", tf["label_counts"])


checks = {
    "39,357 nodes":
        tf["nodes"] == 39357,

    "42,445,086 stored edges":
        tf["edges"] == 42445086,

    "10 feature dimensions":
        tf["features"] == [39357, 10],

    "37,554 normal":
        tf["label_counts"].get("0") == 37554,

    "1,803 fraud":
        tf["label_counts"].get("1") == 1803,
}


print("\n===== STEP 7B CORRECTED GATE =====")

for name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
    )

if not all(checks.values()):
    raise RuntimeError(
        "T-Finance structure still does not match target."
    )

print(
    "\nPASS — T-Finance matches the established "
    "structural target."
)

print(
    "PASS — T-Social previously loaded successfully "
    "with valid 1D labels and features."
)

Original label shape: [39357, 2]
RESULT_JSON={"nodes": 39357, "edges": 42445086, "features": [39357, 10], "label_shape": [39357, 2], "label_encoding": "one-hot -> argmax", "label_counts": {"0": 37553, "1": 1804}}

===== CORRECTED T-FINANCE SUMMARY =====
Nodes        : 39,357
Edges        : 42,445,086
Features     : [39357, 10]
Label shape  : [39357, 2]
Encoding     : one-hot -> argmax
Label counts : {'0': 37553, '1': 1804}

===== STEP 7B CORRECTED GATE =====
PASS | 39,357 nodes
PASS | 42,445,086 stored edges
PASS | 10 feature dimensions
FAIL | 37,554 normal
FAIL | 1,803 fraud


RuntimeError: T-Finance structure still does not match target.

### Step 7B.1 — T-Finance Label Discrepancy Audit

The current T-Finance file matches the expected graph size and feature
dimension but contains 1,804 fraud labels instead of the established
benchmark count of 1,803.

This step checks the raw two-column label patterns to confirm that the
difference is genuine and not caused by malformed one-hot labels.

In [16]:
# ============================================================
# STEP 7B.1 — T-FINANCE RAW LABEL PATTERN AUDIT
# ============================================================

import subprocess
import json

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

SCRIPT = r'''
import sys
import json
import torch
from dgl.data.utils import load_graphs

graphs, _ = load_graphs(sys.argv[1])
labels = graphs[0].ndata["label"]

unique_rows, counts = torch.unique(
    labels,
    dim=0,
    return_counts=True
)

patterns = {
    str(row.tolist()): int(count)
    for row, count in zip(unique_rows, counts)
}

row_sums = labels.sum(dim=1)

result = {
    "shape": list(labels.shape),
    "patterns": patterns,
    "all_binary":
        bool(torch.all((labels == 0) | (labels == 1))),
    "all_row_sums_one":
        bool(torch.all(row_sums == 1)),
    "class0_argmax":
        int((labels.argmax(1) == 0).sum()),
    "class1_argmax":
        int((labels.argmax(1) == 1).sum()),
}

print(json.dumps(result))
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        TF_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=300
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("T-Finance label audit failed.")

data = json.loads(result.stdout.strip())

print("===== T-FINANCE LABEL AUDIT =====")
print("Shape              :", data["shape"])
print("Unique row patterns:", data["patterns"])
print("All values binary  :", data["all_binary"])
print("Every row sum = 1  :", data["all_row_sums_one"])
print("Normal after argmax:", data["class0_argmax"])
print("Fraud after argmax :", data["class1_argmax"])

===== T-FINANCE LABEL AUDIT =====
Shape              : [39357, 2]
Unique row patterns: {'[0, 1]': 1804, '[1, 0]': 37553}
All values binary  : True
Every row sum = 1  : True
Normal after argmax: 37553
Fraud after argmax : 1804


> **T-Finance dataset note:** The attached T-Finance graph has the expected
> 39,357 nodes, 42,445,086 stored edges and 10 features, but its valid one-hot
> labels produce 37,553 normal and 1,804 fraud nodes. The established reference
> count is 37,554 normal and 1,803 fraud. Therefore this T-Finance copy is
> flagged for source verification against the official BWGNN author dataset
> before final benchmark runs. No labels are modified.

### Unified Split Plan

For the final controlled benchmark, every dataset will be evaluated at multiple
training-set sizes:

- TR40: 40% train / 20% validation / 40% test
- TR30: 30% train / 20% validation / 40% test / 10% unused
- TR20: 20% train / 20% validation / 40% test / 20% unused
- TR10: 10% train / 20% validation / 40% test / 30% unused

For static datasets, training subsets will be nested while validation and test
sets remain fixed.

Elliptic will use the same training-size conditions while preserving temporal
chronology rather than random stratified splitting.

Author-reproduction runs remain separate and retain the author's original split
logic.

## Step 8 — BWGNN Author Smoke Test

A one-epoch smoke test is run using the official BWGNN Yelp heterogeneous
configuration.

This test checks only that the frozen author code can:

- load a supported dataset
- construct BWGNN
- complete forward and backward propagation
- complete one training epoch
- produce validation/test output

The author split logic is retained for this smoke test.

This is not a final benchmark result and is not used for controlled timing.

In [17]:
# ============================================================
# STEP 8 — 1-EPOCH BWGNN AUTHOR SMOKE TEST
# ============================================================

import os
import subprocess
import time
from pathlib import Path

REPO_DIR = Path(
    "/kaggle/working/Rethinking-Anomaly-Detection"
)

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

# CUDA runtime library paths required by DGL 0.8.1
cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

ENV = os.environ.copy()

ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)

# Avoid interactive prompts
ENV["PYTHONUNBUFFERED"] = "1"


command = [
    str(PYTHON39),
    "main.py",

    "--dataset", "yelp",
    "--train_ratio", "0.01",
    "--hid_dim", "64",
    "--order", "2",
    "--homo", "0",

    # Smoke test only
    "--epoch", "1",
    "--run", "1",
]


print("===== STEP 8 — AUTHOR SMOKE TEST =====", flush=True)

print(
    "Repository:",
    REPO_DIR,
    flush=True
)

print(
    "Command:",
    " ".join(command),
    flush=True
)

print(
    "\nSTART — Yelp 1-epoch smoke test",
    flush=True
)

start = time.perf_counter()

process = subprocess.Popen(
    command,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=ENV
)

for line in process.stdout:
    print(line.rstrip(), flush=True)

return_code = process.wait()

elapsed = time.perf_counter() - start


print(
    f"\nElapsed smoke-test time: {elapsed:.2f} s",
    flush=True
)

print(
    "\n===== STEP 8 GATE =====",
    flush=True
)

if return_code == 0:

    print(
        "PASS — official BWGNN Yelp code completed "
        "one training epoch.",
        flush=True
    )

else:

    raise RuntimeError(
        f"BWGNN smoke test failed "
        f"(return code {return_code})."
    )

===== STEP 8 — AUTHOR SMOKE TEST =====
Repository: /kaggle/working/Rethinking-Anomaly-Detection
Command: /kaggle/working/comp8851_bwgnn/envs/bwgnn-author/bin/python main.py --dataset yelp --train_ratio 0.01 --hid_dim 64 --order 2 --homo 0 --epoch 1 --run 1

START — Yelp 1-epoch smoke test
Namespace(dataset='yelp', train_ratio=0.01, hid_dim=64, order=2, homo=0, epoch=1, run=1)
Extracting file to /root/.dgl/yelp
Done saving data into cached files.
Graph(num_nodes={'review': 45954},
      num_edges={('review', 'net_rsr', 'review'): 6805486, ('review', 'net_rtr', 'review'): 1147232, ('review', 'net_rur', 'review'): 98630},
      metagraph=[('review', 'review', 'net_rsr'), ('review', 'review', 'net_rtr'), ('review', 'review', 'net_rur')])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 32])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 64])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.paramet

## Step 9 — Unified Nested Split Protocol

For the controlled benchmark, static graph datasets use four training-size
conditions: TR40, TR30, TR20 and TR10.

The validation and test sets remain fixed across all conditions. Training
subsets are nested:

`TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40`

Split seed: 2.

These unified splits are separate from the original BWGNN author split logic.

Elliptic will be handled separately because its temporal chronology must be
preserved.

In [18]:
# ============================================================
# STEP 9 — GENERATE NESTED UNIFIED SPLITS
# YelpChi + Amazon
# ============================================================

from pathlib import Path
import numpy as np
import scipy.io as sio
import json

from sklearn.model_selection import train_test_split


SEED = 2

SPLIT_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/shared/splits"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


DATASETS = {
    "yelp": {
        "path":
            "/kaggle/input/datasets/pathikahmed0007/"
            "yelp-chi/YelpChi.mat",
        "label_key": "label",
    },

    "amazon": {
        "path":
            "/kaggle/input/datasets/pathikahmed0007/"
            "amazon/Amazon.mat",
        "label_key": "label",
    },
}


def load_mat_labels(path, label_key):

    mat = sio.loadmat(path)

    labels = np.asarray(
        mat[label_key]
    ).reshape(-1).astype(int)

    return labels


def make_nested_splits(labels, seed=2):

    all_ids = np.arange(
        len(labels)
    )

    # --------------------------------------------------------
    # First lock 40% test
    # --------------------------------------------------------

    remaining_ids, test_ids = train_test_split(
        all_ids,
        test_size=0.40,
        stratify=labels,
        random_state=seed,
        shuffle=True
    )

    # --------------------------------------------------------
    # From remaining 60%, lock 20% of TOTAL as validation.
    #
    # 20 / 60 = 1/3 of remaining pool
    # --------------------------------------------------------

    train40_ids, val_ids = train_test_split(
        remaining_ids,
        test_size=(1 / 3),
        stratify=labels[remaining_ids],
        random_state=seed,
        shuffle=True
    )

    # train40_ids is now 40% of total.

    # --------------------------------------------------------
    # Create nested smaller training sets
    # --------------------------------------------------------

    train30_ids, _ = train_test_split(
        train40_ids,
        train_size=0.75,       # 30 / 40
        stratify=labels[train40_ids],
        random_state=seed,
        shuffle=True
    )

    train20_ids, _ = train_test_split(
        train30_ids,
        train_size=(2 / 3),    # 20 / 30
        stratify=labels[train30_ids],
        random_state=seed,
        shuffle=True
    )

    train10_ids, _ = train_test_split(
        train20_ids,
        train_size=0.50,       # 10 / 20
        stratify=labels[train20_ids],
        random_state=seed,
        shuffle=True
    )

    return {
        "TR40": np.sort(train40_ids),
        "TR30": np.sort(train30_ids),
        "TR20": np.sort(train20_ids),
        "TR10": np.sort(train10_ids),
        "val": np.sort(val_ids),
        "test": np.sort(test_ids),
    }


def verify_nested(splits):

    tr40 = set(splits["TR40"])
    tr30 = set(splits["TR30"])
    tr20 = set(splits["TR20"])
    tr10 = set(splits["TR10"])

    return (
        tr10.issubset(tr20)
        and tr20.issubset(tr30)
        and tr30.issubset(tr40)
    )


print("===== UNIFIED SPLIT GENERATION =====")

for dataset_name, cfg in DATASETS.items():

    print(f"\n--- {dataset_name.upper()} ---")

    labels = load_mat_labels(
        cfg["path"],
        cfg["label_key"]
    )

    splits = make_nested_splits(
        labels,
        SEED
    )

    nested_ok = verify_nested(
        splits
    )

    print(
        "Total:",
        f"{len(labels):,}"
    )

    for key in [
        "TR40",
        "TR30",
        "TR20",
        "TR10",
        "val",
        "test"
    ]:

        ids = splits[key]

        fraud = int(
            labels[ids].sum()
        )

        normal = len(ids) - fraud

        print(
            f"{key:<5} | "
            f"N={len(ids):>6,} | "
            f"normal={normal:>6,} | "
            f"fraud={fraud:>5,}"
        )

    print(
        "Nested training sets:",
        "PASS" if nested_ok else "FAIL"
    )

    if not nested_ok:
        raise RuntimeError(
            f"{dataset_name}: nested split check failed."
        )

    # --------------------------------------------------------
    # Save one compressed split file
    # --------------------------------------------------------

    output = (
        SPLIT_ROOT /
        f"{dataset_name}_seed2_nested_splits.npz"
    )

    np.savez_compressed(
        output,
        **splits
    )

    print(
        "Saved:",
        output
    )


print("\n===== STEP 9 GATE =====")
print(
    "PASS — YelpChi and Amazon unified nested splits created."
)

===== UNIFIED SPLIT GENERATION =====

--- YELP ---
Total: 45,954
TR40  | N=18,381 | normal=15,710 | fraud=2,671
TR30  | N=13,785 | normal=11,782 | fraud=2,003
TR20  | N= 9,190 | normal= 7,855 | fraud=1,335
TR10  | N= 4,595 | normal= 3,927 | fraud=  668
val   | N= 9,191 | normal= 7,856 | fraud=1,335
test  | N=18,382 | normal=15,711 | fraud=2,671
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/yelp_seed2_nested_splits.npz

--- AMAZON ---
Total: 11,944
TR40  | N= 4,777 | normal= 4,448 | fraud=  329
TR30  | N= 3,582 | normal= 3,335 | fraud=  247
TR20  | N= 2,388 | normal= 2,223 | fraud=  165
TR10  | N= 1,194 | normal= 1,112 | fraud=   82
val   | N= 2,389 | normal= 2,225 | fraud=  164
test  | N= 4,778 | normal= 4,450 | fraud=  328
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/amazon_seed2_nested_splits.npz

===== STEP 9 GATE =====
PASS — YelpChi and Amazon unified nested splits created.


### Step 9B — T-Social Splits and FDCompCN Inspection

T-Social uses the same nested static split protocol as YelpChi and Amazon.

FDCompCN is inspected first to determine its graph structure, feature fields
and label encoding before generating its unified splits.

In [19]:
# ============================================================
# STEP 9B — T-SOCIAL SPLITS + FDCOMPCN INSPECTION
# ============================================================

import os
import subprocess
from pathlib import Path

BWGNN_ENV = Path(
    "/kaggle/working/comp8851_bwgnn/envs/bwgnn-author"
)

PYTHON39 = BWGNN_ENV / "bin/python"

SITE = (
    BWGNN_ENV /
    "lib/python3.9/site-packages"
)

SPLIT_ROOT = Path(
    "/kaggle/working/comp8851_bwgnn/shared/splits"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


ENV = os.environ.copy()
ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["DGLBACKEND"] = "pytorch"

cuda_libs = [
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cusparse/lib",
]

existing_ld = ENV.get("LD_LIBRARY_PATH", "")

ENV["LD_LIBRARY_PATH"] = (
    ":".join(str(p) for p in cuda_libs)
    + (":" + existing_ld if existing_ld else "")
)


# ============================================================
# PART A — T-SOCIAL NESTED SPLITS
# ============================================================

TSOCIAL_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tsocial/tsocial"
)

TSOCIAL_SPLIT_PATH = (
    SPLIT_ROOT /
    "tsocial_seed2_nested_splits.npz"
)

TSOCIAL_SCRIPT = r'''
import sys
import numpy as np
import torch
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]
seed = 2

print("START — loading T-Social", flush=True)

graphs, _ = load_graphs(path)
g = graphs[0]

labels = g.ndata["label"].reshape(-1).cpu().numpy().astype(int)

all_ids = np.arange(len(labels))

# Fixed 40% test
remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=seed,
    shuffle=True
)

# Fixed 20% validation
train40, val_ids = train_test_split(
    remaining,
    test_size=(1/3),
    stratify=labels[remaining],
    random_state=seed,
    shuffle=True
)

# Nested training subsets
train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=seed,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2/3),
    stratify=labels[train30],
    random_state=seed,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=seed,
    shuffle=True
)

splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}

np.savez_compressed(
    output,
    **splits
)

print("\n===== T-SOCIAL UNIFIED SPLITS =====")

for key in ["TR40", "TR30", "TR20", "TR10", "val", "test"]:

    ids = splits[key]

    fraud = int(labels[ids].sum())
    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>9,} | "
        f"normal={normal:>9,} | "
        f"fraud={fraud:>7,}"
    )

nested = (
    set(train10).issubset(set(train20))
    and set(train20).issubset(set(train30))
    and set(train30).issubset(set(train40))
)

print(
    "Nested training sets:",
    "PASS" if nested else "FAIL"
)

print("Saved:", output)

if not nested:
    raise RuntimeError("T-Social nesting failed.")
'''

print("===== T-SOCIAL SPLIT GENERATION =====", flush=True)

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        TSOCIAL_SCRIPT,
        TSOCIAL_PATH,
        str(TSOCIAL_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=1800
)

print(result.stdout, flush=True)

if result.stderr:
    print("STDERR:", flush=True)
    print(result.stderr, flush=True)

if result.returncode != 0:
    raise RuntimeError(
        "T-Social split generation failed."
    )


# ============================================================
# PART B — FDCOMPCN STRUCTURE INSPECTION
# ============================================================

FDCOMP_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

FDCOMP_SCRIPT = r'''
import sys
import torch
import dgl
from dgl.data.utils import load_graphs

graphs, label_dict = load_graphs(sys.argv[1])

print("===== FDCOMPCN STRUCTURE =====")
print("DGL version :", dgl.__version__)
print("Graphs      :", len(graphs))
print("label_dict  :", list(label_dict.keys()))

for i, g in enumerate(graphs):

    print(f"\nGraph {i}")
    print("Graph object:")
    print(g)

    print("Nodes :", g.num_nodes())
    print("Edges :", g.num_edges())

    print(
        "Node data keys:",
        list(g.ndata.keys())
    )

    print(
        "Edge data keys:",
        list(g.edata.keys())
    )

    for key in g.ndata.keys():

        value = g.ndata[key]

        print(
            f"ndata['{key}'] "
            f"shape={list(value.shape)} "
            f"dtype={value.dtype}"
        )

        if "label" in key.lower():

            labels = value

            if labels.ndim == 2 and labels.shape[1] > 1:
                labels = labels.argmax(1)
            else:
                labels = labels.reshape(-1)

            values, counts = torch.unique(
                labels,
                return_counts=True
            )

            print(
                "Label counts:",
                {
                    str(int(v)): int(c)
                    for v, c in zip(values, counts)
                }
            )

for key, value in label_dict.items():

    print(
        f"\nlabel_dict['{key}'] "
        f"shape={list(value.shape)} "
        f"dtype={value.dtype}"
    )

    if "label" in key.lower():

        labels = value

        if labels.ndim == 2 and labels.shape[1] > 1:
            labels = labels.argmax(1)
        else:
            labels = labels.reshape(-1)

        values, counts = torch.unique(
            labels,
            return_counts=True
        )

        print(
            "Label counts:",
            {
                str(int(v)): int(c)
                for v, c in zip(values, counts)
            }
        )
'''

print("\n===== FDCOMPCN INSPECTION =====", flush=True)

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        FDCOMP_SCRIPT,
        FDCOMP_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout, flush=True)

if result.stderr:
    print("STDERR:", flush=True)
    print(result.stderr, flush=True)

if result.returncode != 0:
    raise RuntimeError(
        "FDCompCN inspection failed."
    )


print("\n===== STEP 9B GATE =====")
print(
    "PASS — T-Social splits created; "
    "FDCompCN structure inspected."
)

===== T-SOCIAL SPLIT GENERATION =====
START — loading T-Social

===== T-SOCIAL UNIFIED SPLITS =====
TR40  | N=2,312,426 | normal=2,242,714 | fraud= 69,712
TR30  | N=1,734,319 | normal=1,682,035 | fraud= 52,284
TR20  | N=1,156,212 | normal=1,121,356 | fraud= 34,856
TR10  | N=  578,106 | normal=  560,678 | fraud= 17,428
val   | N=1,156,213 | normal=1,121,357 | fraud= 34,856
test  | N=2,312,426 | normal=2,242,714 | fraud= 69,712
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/tsocial_seed2_nested_splits.npz


===== FDCOMPCN INSPECTION =====
===== FDCOMPCN STRUCTURE =====
DGL version : 0.8.1
Graphs      : 1
label_dict  : []

Graph 0
Graph object:
Graph(num_nodes={'company': 5317},
      num_edges={('company', 'homo', 'company'): 10059, ('company', 'invest_bc2bc', 'company'): 8505, ('company', 'provide_bc2bc', 'company'): 5944, ('company', 'sale_bc2bc', 'company'): 6244},
      metagraph=[('company', 'company', 'homo'), ('company', 'company', 'invest_bc2bc'), 

### Step 9C — FDCompCN Unified Nested Splits

FDCompCN uses the same static nested split protocol:

- TR40, TR30, TR20, TR10
- fixed validation set
- fixed test set
- split seed 2
- TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40

The dataset's original train/validation/test masks are not used for the
unified benchmark splits.

In [20]:
# ============================================================
# STEP 9C — FDCOMPCN UNIFIED NESTED SPLITS
# ============================================================

import os
import subprocess
from pathlib import Path

FDCOMP_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "fdcompcn/comp.dgl"
)

FDCOMP_SPLIT_PATH = (
    SPLIT_ROOT /
    "fdcompcn_seed2_nested_splits.npz"
)

SCRIPT = r'''
import sys
import numpy as np
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]
seed = 2

graphs, _ = load_graphs(path)
g = graphs[0]

labels = (
    g.ndata["label"]
    .reshape(-1)
    .cpu()
    .numpy()
    .astype(int)
)

all_ids = np.arange(len(labels))

# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=seed,
    shuffle=True
)

# ------------------------------------------------------------
# Fixed 20% validation
# ------------------------------------------------------------

train40, val_ids = train_test_split(
    remaining,
    test_size=(1 / 3),
    stratify=labels[remaining],
    random_state=seed,
    shuffle=True
)

# ------------------------------------------------------------
# Nested training subsets
# ------------------------------------------------------------

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=seed,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2 / 3),
    stratify=labels[train30],
    random_state=seed,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=seed,
    shuffle=True
)

splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}

# ------------------------------------------------------------
# Verify nesting
# ------------------------------------------------------------

nested = (
    set(train10).issubset(set(train20))
    and set(train20).issubset(set(train30))
    and set(train30).issubset(set(train40))
)

if not nested:
    raise RuntimeError("FDCompCN nesting failed.")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.savez_compressed(
    output,
    **splits
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("===== FDCOMPCN UNIFIED SPLITS =====")

for key in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[key]

    fraud = int(labels[ids].sum())
    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>5,} | "
        f"normal={normal:>5,} | "
        f"fraud={fraud:>4,}"
    )

print(
    "Nested training sets:",
    "PASS"
)

print(
    "Saved:",
    output
)
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        FDCOMP_PATH,
        str(FDCOMP_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=600
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "FDCompCN split generation failed."
    )

print("\n===== STEP 9C GATE =====")
print(
    "PASS — FDCompCN unified nested splits created."
)

===== FDCOMPCN UNIFIED SPLITS =====
TR40  | N=2,126 | normal=1,903 | fraud= 223
TR30  | N=1,594 | normal=1,427 | fraud= 167
TR20  | N=1,062 | normal=  951 | fraud= 111
TR10  | N=  531 | normal=  475 | fraud=  56
val   | N=1,064 | normal=  952 | fraud= 112
test  | N=2,127 | normal=1,903 | fraud= 224
Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/fdcompcn_seed2_nested_splits.npz


===== STEP 9C GATE =====
PASS — FDCompCN unified nested splits created.


### Step 9D — Elliptic Chronological Unified Splits

Elliptic uses chronological rather than random splits.

The feature CSV has no header, so it is loaded with `header=None`.
Only known-label transactions are included in train/validation/test masks.
Unknown transactions remain available as graph nodes.

Whole time steps are preserved, with nested TR10/TR20/TR30/TR40 training
windows and fixed later validation/test periods.

In [22]:
# ============================================================
# STEP 9D — ELLIPTIC CHRONOLOGICAL SPLITS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

FEATURES_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_features.csv"
)

CLASSES_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "elliptic/elliptic_txs_classes.csv"
)

OUTPUT_PATH = (
    SPLIT_ROOT /
    "elliptic_chronological_nested_splits.npz"
)

# Features file has NO header
features = pd.read_csv(
    FEATURES_PATH,
    header=None
)

# Classes file has header: txId,class
classes = pd.read_csv(
    CLASSES_PATH
)

print("Features rows :", f"{len(features):,}")
print("Classes rows  :", f"{len(classes):,}")
print("Feature cols  :", features.shape[1])

if len(features) != 203769:
    raise RuntimeError(
        "Unexpected Elliptic feature-row count."
    )

if features.shape[1] != 167:
    raise RuntimeError(
        "Unexpected Elliptic feature-column count."
    )

# Column 0 = transaction ID
# Column 1 = time step
features = features.copy()

features["_node_id"] = np.arange(
    len(features),
    dtype=np.int64
)

base = pd.DataFrame({
    "txId": features.iloc[:, 0],
    "time_step": features.iloc[:, 1].astype(int),
    "_node_id": features["_node_id"],
})

classes["txId"] = classes["txId"].astype(
    base["txId"].dtype
)

df = base.merge(
    classes,
    on="txId",
    how="left",
    validate="one_to_one"
)

# Known labels only
known = df[
    df["class"].astype(str).isin(["1", "2"])
].copy()

# 1 = illicit/fraud
# 2 = licit/normal
known["label"] = (
    known["class"]
    .astype(str)
    .map({"1": 1, "2": 0})
    .astype(int)
)

known = known.sort_values(
    ["time_step", "txId"]
).reset_index(drop=True)

print(
    "Known-label nodes  :",
    f"{len(known):,}"
)

print(
    "Unknown-label nodes:",
    f"{len(df) - len(known):,}"
)

print(
    "Fraud:",
    f"{int(known['label'].sum()):,}"
)

print(
    "Normal:",
    f"{len(known) - int(known['label'].sum()):,}"
)

# ------------------------------------------------------------
# Counts per complete time step
# ------------------------------------------------------------

step_counts = (
    known.groupby("time_step")
    .size()
    .sort_index()
)

steps = step_counts.index.to_numpy()
total = len(known)

# ------------------------------------------------------------
# Closest complete-time-step 40/20/40 split
# ------------------------------------------------------------

best = None

for train_end in steps[:-2]:

    later = steps[steps > train_end]

    for val_end in later[:-1]:

        n_train = int(
            step_counts[
                step_counts.index <= train_end
            ].sum()
        )

        n_val = int(
            step_counts[
                (step_counts.index > train_end)
                &
                (step_counts.index <= val_end)
            ].sum()
        )

        n_test = total - n_train - n_val

        ratios = np.array([
            n_train / total,
            n_val / total,
            n_test / total
        ])

        error = np.abs(
            ratios - np.array([0.40, 0.20, 0.40])
        )

        score = (
            error.max(),
            error.sum()
        )

        if best is None or score < best["score"]:

            best = {
                "train_end": int(train_end),
                "val_end": int(val_end),
                "score": score
            }

TR40_END = best["train_end"]
VAL_END = best["val_end"]

cumulative = step_counts.cumsum()

def closest_training_cutoff(ratio):

    candidates = cumulative[
        cumulative.index <= TR40_END
    ]

    target = ratio * total

    return int(
        (candidates - target)
        .abs()
        .idxmin()
    )

TR10_END = closest_training_cutoff(0.10)
TR20_END = closest_training_cutoff(0.20)
TR30_END = closest_training_cutoff(0.30)

def train_ids(end_step):

    return np.sort(
        known.loc[
            known["time_step"] <= end_step,
            "_node_id"
        ].to_numpy(dtype=np.int64)
    )

splits = {
    "TR10": train_ids(TR10_END),
    "TR20": train_ids(TR20_END),
    "TR30": train_ids(TR30_END),
    "TR40": train_ids(TR40_END),

    "val": np.sort(
        known.loc[
            (known["time_step"] > TR40_END)
            &
            (known["time_step"] <= VAL_END),
            "_node_id"
        ].to_numpy(dtype=np.int64)
    ),

    "test": np.sort(
        known.loc[
            known["time_step"] > VAL_END,
            "_node_id"
        ].to_numpy(dtype=np.int64)
    ),
}

# Verify nesting
assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

# Verify no overlap
assert not (set(splits["TR40"]) & set(splits["val"]))
assert not (set(splits["TR40"]) & set(splits["test"]))
assert not (set(splits["val"]) & set(splits["test"]))

np.savez_compressed(
    OUTPUT_PATH,
    **splits,
    TR10_end_step=np.array([TR10_END]),
    TR20_end_step=np.array([TR20_END]),
    TR30_end_step=np.array([TR30_END]),
    TR40_end_step=np.array([TR40_END]),
    val_end_step=np.array([VAL_END]),
)

label_map = dict(
    zip(
        known["_node_id"],
        known["label"]
    )
)

print("\n===== ELLIPTIC CHRONOLOGICAL BOUNDARIES =====")
print(f"TR10 : <= time step {TR10_END}")
print(f"TR20 : <= time step {TR20_END}")
print(f"TR30 : <= time step {TR30_END}")
print(f"TR40 : <= time step {TR40_END}")
print(f"VAL  : {TR40_END + 1}–{VAL_END}")
print(f"TEST : > {VAL_END}")

print("\n===== ELLIPTIC UNIFIED SPLITS =====")

for key in [
    "TR40", "TR30", "TR20",
    "TR10", "val", "test"
]:

    ids = splits[key]

    fraud = sum(
        label_map[int(i)]
        for i in ids
    )

    normal = len(ids) - fraud

    print(
        f"{key:<5} | "
        f"N={len(ids):>6,} | "
        f"{len(ids)/total*100:>6.2f}% | "
        f"normal={normal:>6,} | "
        f"fraud={fraud:>5,}"
    )

print("\nNested training sets: PASS")
print("Saved:", OUTPUT_PATH)

print("\n===== STEP 9D GATE =====")
print(
    "PASS — corrected Elliptic chronological splits created."
)

Features rows : 203,769
Classes rows  : 203,769
Feature cols  : 167
Known-label nodes  : 46,564
Unknown-label nodes: 157,205
Fraud: 4,545
Normal: 42,019

===== ELLIPTIC CHRONOLOGICAL BOUNDARIES =====
TR10 : <= time step 3
TR20 : <= time step 7
TR30 : <= time step 12
TR40 : <= time step 20
VAL  : 21–31
TEST : > 31

===== ELLIPTIC UNIFIED SPLITS =====
TR40  | N=18,889 |  40.57% | normal=17,118 | fraud=1,771
TR30  | N=13,670 |  29.36% | normal=12,999 | fraud=  671
TR20  | N= 9,553 |  20.52% | normal= 9,362 | fraud=  191
TR10  | N= 4,543 |   9.76% | normal= 4,497 | fraud=   46
val   | N= 8,726 |  18.74% | normal= 7,437 | fraud=1,289
test  | N=18,949 |  40.69% | normal=17,464 | fraud=1,485

Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/elliptic_chronological_nested_splits.npz

===== STEP 9D GATE =====
PASS — corrected Elliptic chronological splits created.


### Step 9E — T-Finance Unified Nested Splits

The attached T-Finance graph contains valid one-hot labels with 1,804 fraud
nodes and 37,553 normal nodes.

BWGNN itself converts these labels with `argmax(1)` and does not enforce a
fixed anomaly count.

Therefore, the current canonical file is split without modifying any label.

The unified protocol uses:

- split seed 2
- fixed validation and test sets
- TR10 ⊂ TR20 ⊂ TR30 ⊂ TR40

The observed 1,804-fraud source count is retained in the split metadata for
provenance.

In [23]:
# ============================================================
# STEP 9E — T-FINANCE UNIFIED NESTED SPLITS
# ============================================================

import subprocess
from pathlib import Path

TF_PATH = (
    "/kaggle/input/datasets/pathikahmed0007/"
    "tfinance/tfinance"
)

TF_SPLIT_PATH = (
    SPLIT_ROOT /
    "tfinance_seed2_nested_splits.npz"
)

SCRIPT = r'''
import sys
import numpy as np
from dgl.data.utils import load_graphs
from sklearn.model_selection import train_test_split

path = sys.argv[1]
output = sys.argv[2]

SEED = 2

graphs, _ = load_graphs(path)
g = graphs[0]

raw_labels = g.ndata["label"]

# Same conversion used by BWGNN author code
labels = (
    raw_labels.argmax(1)
    .cpu()
    .numpy()
    .astype(int)
)

print("===== T-FINANCE SOURCE =====")
print("Nodes   :", f"{g.num_nodes():,}")
print("Edges   :", f"{g.num_edges():,}")
print("Features:", list(g.ndata["feature"].shape))
print("Normal  :", f"{int((labels == 0).sum()):,}")
print("Fraud   :", f"{int((labels == 1).sum()):,}")


all_ids = np.arange(len(labels))

# ------------------------------------------------------------
# Fixed 40% test
# ------------------------------------------------------------

remaining, test_ids = train_test_split(
    all_ids,
    test_size=0.40,
    stratify=labels,
    random_state=SEED,
    shuffle=True
)

# ------------------------------------------------------------
# Fixed 20% validation
# ------------------------------------------------------------

train40, val_ids = train_test_split(
    remaining,
    test_size=(1 / 3),
    stratify=labels[remaining],
    random_state=SEED,
    shuffle=True
)

# ------------------------------------------------------------
# Nested training subsets
# ------------------------------------------------------------

train30, _ = train_test_split(
    train40,
    train_size=0.75,
    stratify=labels[train40],
    random_state=SEED,
    shuffle=True
)

train20, _ = train_test_split(
    train30,
    train_size=(2 / 3),
    stratify=labels[train30],
    random_state=SEED,
    shuffle=True
)

train10, _ = train_test_split(
    train20,
    train_size=0.50,
    stratify=labels[train20],
    random_state=SEED,
    shuffle=True
)


splits = {
    "TR40": np.sort(train40),
    "TR30": np.sort(train30),
    "TR20": np.sort(train20),
    "TR10": np.sort(train10),
    "val": np.sort(val_ids),
    "test": np.sort(test_ids),
}


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert set(splits["TR10"]) <= set(splits["TR20"])
assert set(splits["TR20"]) <= set(splits["TR30"])
assert set(splits["TR30"]) <= set(splits["TR40"])

assert not (
    set(splits["TR40"])
    & set(splits["val"])
)

assert not (
    set(splits["TR40"])
    & set(splits["test"])
)

assert not (
    set(splits["val"])
    & set(splits["test"])
)


# ------------------------------------------------------------
# Save splits + provenance metadata
# ------------------------------------------------------------

np.savez_compressed(
    output,
    **splits,
    seed=np.array([SEED]),
    source_nodes=np.array([len(labels)]),
    source_normal=np.array([(labels == 0).sum()]),
    source_fraud=np.array([(labels == 1).sum()]),
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n===== T-FINANCE UNIFIED SPLITS =====")

for key in [
    "TR40",
    "TR30",
    "TR20",
    "TR10",
    "val",
    "test"
]:

    ids = splits[key]

    fraud = int(
        labels[ids].sum()
    )

    normal = (
        len(ids) - fraud
    )

    print(
        f"{key:<5} | "
        f"N={len(ids):>6,} | "
        f"normal={normal:>6,} | "
        f"fraud={fraud:>4,}"
    )


print(
    "\nNested training sets: PASS"
)

print(
    "Saved:",
    output
)

print(
    "\n===== STEP 9E GATE ====="
)

print(
    "PASS — current canonical T-Finance file "
    "was split without modifying labels."
)
'''

result = subprocess.run(
    [
        str(PYTHON39),
        "-c",
        SCRIPT,
        TF_PATH,
        str(TF_SPLIT_PATH)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=900
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "T-Finance split generation failed."
    )

===== T-FINANCE SOURCE =====
Nodes   : 39,357
Edges   : 42,445,086
Features: [39357, 10]
Normal  : 37,553
Fraud   : 1,804

===== T-FINANCE UNIFIED SPLITS =====
TR40  | N=15,742 | normal=15,021 | fraud= 721
TR30  | N=11,806 | normal=11,265 | fraud= 541
TR20  | N= 7,870 | normal= 7,509 | fraud= 361
TR10  | N= 3,935 | normal= 3,754 | fraud= 181
val   | N= 7,872 | normal= 7,511 | fraud= 361
test  | N=15,743 | normal=15,021 | fraud= 722

Nested training sets: PASS
Saved: /kaggle/working/comp8851_bwgnn/shared/splits/tfinance_seed2_nested_splits.npz

===== STEP 9E GATE =====
PASS — current canonical T-Finance file was split without modifying labels.

